In [4]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

In [5]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.cleaning import prepare_dataset

raw_path = PROJECT_ROOT / "data" / "raw" / "Raw_A_Z_medicines_dataset_of_India.csv"

raw_df = pd.read_csv(raw_path)

clean_df = prepare_dataset(raw_df)

print("Raw dataset shape:", raw_df.shape)
print("Cleaned dataset shape:", clean_df.shape)

display(clean_df.head(10))

Raw dataset shape: (253973, 9)
Cleaned dataset shape: (253973, 11)


,medicine_id,name,price,is_discontinued,manufacturer,dosage_form,pack_size,composition1,composition2,price_original,price_status
0,1,Augmentin 625 Duo Tablet,223.42,False,Glaxo SmithKline Pharmaceuticals Ltd,allopathy,strip of 10 tablets,Amoxycillin (500mg),Clavulanic Acid (125mg),223.42,valid
1,2,Azithral 500 Tablet,132.36,False,Alembic Pharmaceuticals Ltd,allopathy,strip of 5 tablets,Azithromycin (500mg),<NA>,132.36,valid
2,3,Ascoril LS Syrup,118.00,False,Glenmark Pharmaceuticals Ltd,allopathy,bottle of 100 ml Syrup,Ambroxol (30mg/5ml),Levosalbutamol (1mg/5ml),118.00,valid
3,4,Allegra 120mg Tablet,218.81,False,Sanofi India Ltd,allopathy,strip of 10 tablets,Fexofenadine (120mg),<NA>,218.81,valid
4,5,Avil 25 Tablet,10.96,False,Sanofi India Ltd,allopathy,strip of 15 tablets,Pheniramine (25mg),<NA>,10.96,valid
5,6,Allegra-M Tablet,241.48,False,Sanofi India Ltd,allopathy,strip of 10 tablets,Montelukast (10mg),Fexofenadine (120mg),241.48,valid
6,7,Amoxyclav 625 Tablet,223.27,False,Abbott,allopathy,strip of 10 tablets,Amoxycillin (500mg),Clavulanic Acid (125mg),223.27,valid
7,8,Azee 500 Tablet,132.38,False,Cipla Ltd,allopathy,strip of 5 tablets,Azithromycin (500mg),<NA>,132.38,valid
8,9,Atarax 25mg Tablet,85.50,False,Dr Reddy's Laboratories Ltd,allopathy,strip of 15 tablets,Hydroxyzine (25mg),<NA>,85.50,valid
9,10,Ascoril D Plus Syrup Sugar Free,129.00,False,Glenmark Pharmaceuticals Ltd,allopathy,bottle of 100 ml Syrup,Phenylephrine (5mg),Chlorpheniramine Maleate (2mg),129.00,valid


In [ ]:
print("Dataset shape:", clean_df.shape)

print("\nCleaned Price Statistics")
display(clean_df["price"].describe())

print("\nPrice Status Distribution")
display(clean_df["price_status"].value_counts(dropna=False))

print("\nInvalid Price Records")

invalid_prices = clean_df[clean_df["price_status"] != "valid"]

display(invalid_prices[ ["medicine_id", "name", "price_original", "price_status"]].head(20))


print("\nMissing Values After Cleaning")
display(clean_df.isna().sum().sort_values(ascending=False))

print("\nOriginal raw dataset shape:", raw_df.shape)


Dataset shape: (253973, 11)

Cleaned Price Statistics


count    253969.000000
mean        270.535105
std        3029.607802
min           1.050000
25%          48.000000
50%          79.000000
75%         140.000000
max      436000.000000
Name: price, dtype: float64


Price Status Distribution


price_status
valid    253969
zero          4
Name: count, dtype: int64


Invalid Price Records


,medicine_id,name,price_original,price_status
35090,35091,Covishield Vaccine,0.0,zero
37004,37005,Covaxin Vaccine,0.0,zero
141999,142000,Moderna Vaccine,0.0,zero
204460,204461,Sputnik V Vaccine,0.0,zero



Missing Values After Cleaning


composition2       141802
price                   4
medicine_id             0
name                    0
is_discontinued         0
dosage_form             0
manufacturer            0
pack_size               0
composition1            0
price_original          0
price_status            0
dtype: int64


Original raw dataset shape: (253973, 9)


In [8]:
import sys
import importlib
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


import src.cleaning as cleaning
importlib.reload(cleaning)


from src.cleaning import (
    parse_strength,
    parse_ingredient,
    parse_medicine_compositions
)

print("All parser functions imported successfully!")

All parser functions imported successfully!


In [ ]:
test_values = [
    "500mg",
    "30mg/5ml",
    "2% w/w",
    "1% w/v",
    "60000IU",
    "1500mcg",
    "60Million spores",
]

for value in test_values:
    print(value, "->", parse_strength(value))


500mg -> {'strength_value': 500.0, 'strength_unit': 'mg', 'denominator_value': None, 'denominator_unit': None, 'strength_basis': None, 'strength_qualifier': None, 'parse_status': 'parsed'}
30mg/5ml -> {'strength_value': 30.0, 'strength_unit': 'mg', 'denominator_value': 5.0, 'denominator_unit': 'ml', 'strength_basis': None, 'strength_qualifier': None, 'parse_status': 'parsed'}
2% w/w -> {'strength_value': 2.0, 'strength_unit': '%', 'denominator_value': None, 'denominator_unit': None, 'strength_basis': 'w/w', 'strength_qualifier': None, 'parse_status': 'parsed'}
1% w/v -> {'strength_value': 1.0, 'strength_unit': '%', 'denominator_value': None, 'denominator_unit': None, 'strength_basis': 'w/v', 'strength_qualifier': None, 'parse_status': 'parsed'}
60000IU -> {'strength_value': 60000.0, 'strength_unit': 'IU', 'denominator_value': None, 'denominator_unit': None, 'strength_basis': None, 'strength_qualifier': None, 'parse_status': 'parsed'}
1500mcg -> {'strength_value': 1500.0, 'strength_unit

In [9]:

clean_df["parsed_ingredients"] = clean_df.apply(parse_medicine_compositions,axis=1)


clean_df["ingredient_count"] = (clean_df["parsed_ingredients"].apply(len))

print("Ingredient count distribution")

display(clean_df["ingredient_count"].value_counts().sort_index())

ingredient_records = []

for _, row in clean_df.iterrows():
    for ingredient in row["parsed_ingredients"]:
        ingredient_records.append({
            "medicine_id": row["medicine_id"],
            **ingredient
        })

ingredients_df = pd.DataFrame(ingredient_records)

print("\nTotal parsed ingredient records:", len(ingredients_df))

print("\nParsing status distribution:")

display(ingredients_df["parse_status"].value_counts(dropna=False))

print("\nParsing status percentages:")

display((ingredients_df["parse_status"].value_counts(normalize=True, dropna=False) * 100).round(2))

print("\nSample parsed ingredients:")

display(ingredients_df.head(20))

Ingredient count distribution


ingredient_count
1    141802
2    112171
Name: count, dtype: int64


Total parsed ingredient records: 366144

Parsing status distribution:


parse_status
parsed               358858
unparsed_strength      7286
Name: count, dtype: int64


Parsing status percentages:


parse_status
parsed               98.01
unparsed_strength     1.99
Name: proportion, dtype: float64


Sample parsed ingredients:


,medicine_id,raw_composition,ingredient,strength_raw,strength_value,strength_unit,denominator_value,denominator_unit,strength_basis,strength_qualifier,parse_status,source_column
0,1,Amoxycillin (500mg),Amoxycillin,500mg,500.000,mg,NaN,NaN,NaN,NaN,parsed,composition1
1,1,Clavulanic Acid (125mg),Clavulanic Acid,125mg,125.000,mg,NaN,NaN,NaN,NaN,parsed,composition2
2,2,Azithromycin (500mg),Azithromycin,500mg,500.000,mg,NaN,NaN,NaN,NaN,parsed,composition1
3,3,Ambroxol (30mg/5ml),Ambroxol,30mg/5ml,30.000,mg,5.0,ml,NaN,NaN,parsed,composition1
4,3,Levosalbutamol (1mg/5ml),Levosalbutamol,1mg/5ml,1.000,mg,5.0,ml,NaN,NaN,parsed,composition2
5,4,Fexofenadine (120mg),Fexofenadine,120mg,120.000,mg,NaN,NaN,NaN,NaN,parsed,composition1
6,5,Pheniramine (25mg),Pheniramine,25mg,25.000,mg,NaN,NaN,NaN,NaN,parsed,composition1
7,6,Montelukast (10mg),Montelukast,10mg,10.000,mg,NaN,NaN,NaN,NaN,parsed,composition1
8,6,Fexofenadine (120mg),Fexofenadine,120mg,120.000,mg,NaN,NaN,NaN,NaN,parsed,composition2
9,7,Amoxycillin (500mg),Amoxycillin,500mg,500.000,mg,NaN,NaN,NaN,NaN,parsed,composition1


In [18]:

test_compositions = [
    "Amoxycillin (500mg)",
    "Ambroxol (30mg/5ml)",
    "Ketoconazole (2% w/w)",
    "Vitamin D3 (60000IU)",
    "Lactobacillus (60Million spores)",
]

for composition in test_compositions:
    print("\n", composition)
    print(parse_ingredient(composition))



 Amoxycillin (500mg)
{'raw_composition': 'Amoxycillin (500mg)', 'ingredient': 'Amoxycillin', 'strength_raw': '500mg', 'strength_value': 500.0, 'strength_unit': 'mg', 'denominator_value': None, 'denominator_unit': None, 'strength_basis': None, 'strength_qualifier': None, 'parse_status': 'parsed'}

 Ambroxol (30mg/5ml)
{'raw_composition': 'Ambroxol (30mg/5ml)', 'ingredient': 'Ambroxol', 'strength_raw': '30mg/5ml', 'strength_value': 30.0, 'strength_unit': 'mg', 'denominator_value': 5.0, 'denominator_unit': 'ml', 'strength_basis': None, 'strength_qualifier': None, 'parse_status': 'parsed'}

 Ketoconazole (2% w/w)
{'raw_composition': 'Ketoconazole (2% w/w)', 'ingredient': 'Ketoconazole', 'strength_raw': '2% w/w', 'strength_value': 2.0, 'strength_unit': '%', 'denominator_value': None, 'denominator_unit': None, 'strength_basis': 'w/w', 'strength_qualifier': None, 'parse_status': 'parsed'}

 Vitamin D3 (60000IU)
{'raw_composition': 'Vitamin D3 (60000IU)', 'ingredient': 'Vitamin D3', 'strength

In [10]:

unparsed = ingredients_df[ingredients_df["parse_status"] != "parsed"].copy()

print("Total unparsed records:", len(unparsed))

print("\nMost common unparsed strength formats:")

display(unparsed["strength_raw"].value_counts(dropna=False).head(40))


print("\nSample unparsed composition records:")

display(
    unparsed[
        [
            "medicine_id",
            "raw_composition",
            "ingredient",
            "strength_raw",
            "parse_status",
        ]
    ].head(40)
)

Total unparsed records: 7286

Most common unparsed strength formats:


strength_raw
NA                   5377
2mg/ml                154
5mg/ml                114
1mg/ml                110
100000AU              109
5Billion Spores        92
100mg/ml               92
2.5Billion Spores      81
25mg/ml                79
50mg/ml                70
2.5mg/ml               66
20mg/ml                65
10mg/ml                56
75mg/ml                55
7.5mg/ml               43
0.25mg/ml              40
250mg/ml               38
4mg/ml                 29
40mg/ml                27
50000AU                25
2Billion Spores        24
100IU/ml               23
75i.u                  22
150mg/ml               20
150i.u                 20
3mg/ml                 16
200mg/ml               16
5LF                    15
125mg/ml               13
500mcg/ml              12
0.75mg/ml              12
50mcg/ml               12
60mg/ml                10
10mg/gm                10
2.5Billion cells        9
1000ccid50              9
0.3mg/ml                8
100000units             8


Sample unparsed composition records:


,medicine_id,raw_composition,ingredient,strength_raw,parse_status
83,60,Ambroxol (7.5mg/ml),Ambroxol,7.5mg/ml,unparsed_strength
84,60,Levosalbutamol (0.25mg/ml),Levosalbutamol,0.25mg/ml,unparsed_strength
135,97,Anti Rh D Immunoglobulin (300mcg/ml),Anti Rh D Immunoglobulin,300mcg/ml,unparsed_strength
214,150,Adrenaline (NA),Adrenaline,NA,unparsed_strength
276,192,Azithromycin (20mg/ml),Azithromycin,20mg/ml,unparsed_strength
295,205,Guaifenesin (NA),Guaifenesin,NA,unparsed_strength
296,205,Bromhexine (NA),Bromhexine,NA,unparsed_strength
471,326,Ambroxol (NA),Ambroxol,NA,unparsed_strength
472,326,Levosalbutamol (NA),Levosalbutamol,NA,unparsed_strength
492,341,Disodium Hydrogen Citrate (NA),Disodium Hydrogen Citrate,NA,unparsed_strength


In [12]:

clean_df["parsed_ingredients"] = clean_df.apply(cleaning.parse_medicine_compositions,axis=1)

ingredient_records = []

for _, row in clean_df.iterrows():
    for ingredient in row["parsed_ingredients"]:
        ingredient_records.append({
            "medicine_id": row["medicine_id"],
            **ingredient
        })

ingredients_df = pd.DataFrame(ingredient_records)

print("Total ingredient records:", len(ingredients_df))

status_counts = ingredients_df["parse_status"].value_counts(dropna=False)

display(status_counts)

print("\nPercentages:")
display((status_counts / len(ingredients_df) * 100).round(2))

print("\nMost common remaining unparsed formats:")

remaining_unparsed = ingredients_df[ingredients_df["parse_status"] == "unparsed_strength"]

display(remaining_unparsed["strength_raw"].value_counts(dropna=False).head(30))

Total ingredient records: 366144


parse_status
parsed               358858
unparsed_strength      7286
Name: count, dtype: int64


Percentages:


parse_status
parsed               98.01
unparsed_strength     1.99
Name: count, dtype: float64


Most common remaining unparsed formats:


strength_raw
NA                   5377
2mg/ml                154
5mg/ml                114
1mg/ml                110
100000AU              109
5Billion Spores        92
100mg/ml               92
2.5Billion Spores      81
25mg/ml                79
50mg/ml                70
2.5mg/ml               66
20mg/ml                65
10mg/ml                56
75mg/ml                55
7.5mg/ml               43
0.25mg/ml              40
250mg/ml               38
4mg/ml                 29
40mg/ml                27
50000AU                25
2Billion Spores        24
100IU/ml               23
75i.u                  22
150mg/ml               20
150i.u                 20
3mg/ml                 16
200mg/ml               16
5LF                    15
125mg/ml               13
500mcg/ml              12
Name: count, dtype: int64

In [ ]:

clean_df["parsed_ingredients"] = clean_df.apply(cleaning.parse_medicine_compositions,axis=1)

ingredient_records = []

for _, row in clean_df.iterrows():
    for ingredient in row["parsed_ingredients"]:
        ingredient_records.append({
            "medicine_id": row["medicine_id"],
            **ingredient
        })

ingredients_df = pd.DataFrame(ingredient_records)


print("Total ingredient records:", len(ingredients_df))

print("\nParsing status:")
display(ingredients_df["parse_status"].value_counts())

print("\nParsing percentages:")
display((ingredients_df["parse_status"].value_counts(normalize=True) * 100).round(2))


remaining_unparsed = ingredients_df[ingredients_df["parse_status"] == "unparsed_strength"]

print("\nTop 30 remaining unrecognized strengths:")
display(remaining_unparsed["strength_raw"].value_counts(dropna=False).head(30))


print("\nActual dataset verification:")
display(
    ingredients_df[
        ingredients_df["strength_raw"].isin(
            ["2mg/ml", "5Billion Spores", "NA"]
        )
    ][
        [
            "raw_composition",
            "strength_raw",
            "strength_value",
            "strength_unit",
            "denominator_value",
            "denominator_unit",
            "parse_status"
        ]
    ].head(15)
)

Total ingredient records: 366144

Parsing status:


parse_status
parsed               360656
missing_strength       5377
unparsed_strength       111
Name: count, dtype: int64


Parsing percentages:


parse_status
parsed               98.50
missing_strength      1.47
unparsed_strength     0.03
Name: proportion, dtype: float64


Top 30 remaining unrecognized strengths:


strength_raw
5LF                    15
1000ccid50              9
60millionspores         7
25LF                    5
3miu                    5
5000ccid50              4
6Lac units              3
40D - Antigen units     3
8D - Antigen units      3
100000FFU               3
0.2CU                   2
2.5billioncells         2
2000PFU                 2
300u/ml                 2
300mg I/ml              2
22.5LF                  1
20millionspores         1
6.5ccid50               1
200mcg/spray            1
1CU                     1
20millioncells          1
2% w/w/1gm              1
20LF                    1
2520LF                  1
57.5LF                  1
.75% w/w                1
10000PFU                1
40000PFU                1
27.5mcg/spray           1
33.6MIU                 1
Name: count, dtype: int64


Actual dataset verification:


,raw_composition,strength_raw,strength_value,strength_unit,denominator_value,denominator_unit,parse_status
214,Adrenaline (NA),NA,NaN,NaN,NaN,NaN,missing_strength
295,Guaifenesin (NA),NA,NaN,NaN,NaN,NaN,missing_strength
296,Bromhexine (NA),NA,NaN,NaN,NaN,NaN,missing_strength
471,Ambroxol (NA),NA,NaN,NaN,NaN,NaN,missing_strength
472,Levosalbutamol (NA),NA,NaN,NaN,NaN,NaN,missing_strength
492,Disodium Hydrogen Citrate (NA),NA,NaN,NaN,NaN,NaN,missing_strength
507,Chlorpheniramine Maleate (2mg/ml),2mg/ml,2.0,mg,1.0,ml,parsed
608,Methylcobalamin (NA),NA,NaN,NaN,NaN,NaN,missing_strength
683,Chlorpheniramine Maleate (NA),NA,NaN,NaN,NaN,NaN,missing_strength
684,Phenylephrine (NA),NA,NaN,NaN,NaN,NaN,missing_strength


In [16]:

unparsed = ingredients_df[ingredients_df["parse_status"] == "unparsed_strength"].copy()

print("Total unparsed records:", len(unparsed))
print("Unique unrecognized formats:", unparsed["strength_raw"].nunique())


display(unparsed["strength_raw"].value_counts(dropna=False).to_frame("count"))

display(
    unparsed[
        [
            "medicine_id",
            "raw_composition",
            "ingredient",
            "strength_raw"
        ]
    ].sort_values("strength_raw")
)

Total unparsed records: 111
Unique unrecognized formats: 59


,count
strength_raw,
5LF,15
1000ccid50,9
60millionspores,7
25LF,5
3miu,5
5000ccid50,4
6Lac units,3
40D - Antigen units,3
8D - Antigen units,3


,medicine_id,raw_composition,ingredient,strength_raw
152171,105301,Clobetasol (.05% w/w),Clobetasol,.05% w/w
254649,177240,Clonazepam (.5mg),Clonazepam,.5mg
88756,61149,Ofloxacin (.75% w/w),Ofloxacin,.75% w/w
37298,25771,Hemocoagulase (0.2CU),Hemocoagulase,0.2CU
54077,37375,Hemocoagulase (0.2CU),Hemocoagulase,0.2CU
...,...,...,...,...
150170,103843,Freeze-dried Live Attenuated Hepatitis A Vacci...,Freeze-dried Live Attenuated Hepatitis A Vaccine,720ELISA
156040,108174,Poliomyelitis Virus Type 2 (8D - Antigen units),Poliomyelitis Virus Type 2,8D - Antigen units
251158,174805,Poliomyelitis Virus Type 2 (8D - Antigen units),Poliomyelitis Virus Type 2,8D - Antigen units
257846,179450,Poliomyelitis Virus Type 2 (8D - Antigen units),Poliomyelitis Virus Type 2,8D - Antigen units


In [21]:
print("Loaded module:", cleaning.__file__)

Loaded module: d:\Engsci_Study_materials\Subject_Index\Projects\cheaper_medicine_finder\src\cleaning.py


In [22]:
test_values = [
    ".05% w/w",
    ".75% w/w",
    ".5mg",
    "60millionspores",
    "2.5billioncells",
    "200mcg/spray",
    "250mcg/spray",
    "300u/ml",
    "3500u/1g",
]

for value in test_values:
    result = cleaning.parse_strength(value)
    print(f"{value:25} -> {result['parse_status']}")

.05% w/w                  -> unparsed_strength
.75% w/w                  -> unparsed_strength
.5mg                      -> unparsed_strength
60millionspores           -> unparsed_strength
2.5billioncells           -> unparsed_strength
200mcg/spray              -> unparsed_strength
250mcg/spray              -> unparsed_strength
300u/ml                   -> unparsed_strength
3500u/1g                  -> unparsed_strength


In [26]:
import inspect
from pathlib import Path
import src.cleaning as cleaning

print("Function name:", cleaning.parse_strength.__name__)
print("Function first line:", cleaning.parse_strength.__code__.co_firstlineno)
print("Function source file:", inspect.getsourcefile(cleaning.parse_strength))

print("\nRelevant definitions in cleaning.py:")

file_path = Path(cleaning.__file__)

for line_number, line in enumerate(
    file_path.read_text(encoding="utf-8").splitlines(),
    start=1
):
    if any(term in line for term in [
        "STRENGTH_PATTERN =",
        "MULTIPLIER_PATTERN =",
        "def parse_strength",
        "def prepare_dataset"
    ]):
        print(f"Line {line_number}: {line.strip()}")

Function name: parse_strength
Function first line: 115
Function source file: d:\Engsci_Study_materials\Subject_Index\Projects\cheaper_medicine_finder\src\cleaning.py

Relevant definitions in cleaning.py:
Line 56: def prepare_dataset(raw_df):
Line 82: STRENGTH_PATTERN = re.compile(
Line 102: MULTIPLIER_PATTERN = re.compile(
Line 117: def parse_strength(strength_text):


In [27]:
print(cleaning.parse_strength("500mg"))
print(cleaning.parse_strength(".5mg"))
print(cleaning.parse_strength("60millionspores"))

{'strength_value': 500.0, 'strength_unit': 'mg', 'denominator_value': None, 'denominator_unit': None, 'strength_basis': None, 'strength_qualifier': None, 'parse_status': 'parsed'}
{'strength_value': None, 'strength_unit': None, 'denominator_value': None, 'denominator_unit': None, 'strength_basis': None, 'strength_qualifier': None, 'parse_status': 'unparsed_strength'}
{'strength_value': None, 'strength_unit': None, 'denominator_value': None, 'denominator_unit': None, 'strength_basis': None, 'strength_qualifier': None, 'parse_status': 'unparsed_strength'}


In [30]:
from pathlib import Path
print(
    Path(cleaning.__file__).read_text(encoding="utf-8").splitlines()[81:115]
)

['', 'STRENGTH_PATTERN = re.compile(', '    r"""', '    ^\\s*', '    (?P<value>(?:\\d+(?:\\.\\d*)?|\\.\\d+))', '    \\s*', '    (?P<unit>mcg|µg|ug|mg|gm|g|i\\.u\\.?|iu|au|units?|u|ml|%)', '    \\s*', '    (?:', '        /', '        \\s*', '        (?:(?P<denominator_value>(?:\\d+(?:\\.\\d*)?|\\.\\d+))\\s*)?', '        (?P<denominator_unit>mcg|µg|ug|mg|gm|g|ml|spray)', '    )?', '    \\s*', '    (?P<basis>w/w|w/v|v/v)?', '    \\s*$', '    """,', '    flags=re.IGNORECASE | re.VERBOSE,', ')', '', 'MULTIPLIER_PATTERN = re.compile(', '    r"""', '    ^\\s*', '    (?P<value>(?:\\d+(?:\\.\\d*)?|\\.\\d+))', '    \\s*', '    (?P<unit>million|billion)', '    \\s*', '    (?P<qualifier>.+?)', '    \\s*$', '    """,', '    flags=re.IGNORECASE | re.VERBOSE,', ')', '']


In [31]:
import inspect
import importlib
import src.cleaning as cleaning

importlib.reload(cleaning)

# Test the regex directly, without calling parse_strength()
tests = [
    (".05% w/w", cleaning.STRENGTH_PATTERN),
    (".5mg", cleaning.STRENGTH_PATTERN),
    ("200mcg/spray", cleaning.STRENGTH_PATTERN),
    ("300u/ml", cleaning.STRENGTH_PATTERN),
    ("3500u/1g", cleaning.STRENGTH_PATTERN),
    ("60millionspores", cleaning.MULTIPLIER_PATTERN),
    ("2.5billioncells", cleaning.MULTIPLIER_PATTERN),
]

for value, pattern in tests:
    match = pattern.fullmatch(value)
    print(f"{value:25} ->", match.groupdict() if match else "NO MATCH")

print("\n--- Actual parse_strength source ---")
print(inspect.getsource(cleaning.parse_strength))

.05% w/w                  -> {'value': '.05', 'unit': '%', 'denominator_value': None, 'denominator_unit': None, 'basis': 'w/w'}
.5mg                      -> {'value': '.5', 'unit': 'mg', 'denominator_value': None, 'denominator_unit': None, 'basis': None}
200mcg/spray              -> {'value': '200', 'unit': 'mcg', 'denominator_value': None, 'denominator_unit': 'spray', 'basis': None}
300u/ml                   -> {'value': '300', 'unit': 'u', 'denominator_value': None, 'denominator_unit': 'ml', 'basis': None}
3500u/1g                  -> {'value': '3500', 'unit': 'u', 'denominator_value': '1', 'denominator_unit': 'g', 'basis': None}
60millionspores           -> {'value': '60', 'unit': 'million', 'qualifier': 'spores'}
2.5billioncells           -> {'value': '2.5', 'unit': 'billion', 'qualifier': 'cells'}

--- Actual parse_strength source ---
def parse_strength(strength_text):
    if pd.isna(strength_text):
        return {
            "strength_value": None,
            "strength_unit": 

In [ ]:

print(
    "Same strength regex:",
    cleaning.parse_strength.__globals__["STRENGTH_PATTERN"]
    is cleaning.STRENGTH_PATTERN
)

print(
    "Same multiplier regex:",
    cleaning.parse_strength.__globals__["MULTIPLIER_PATTERN"]
    is cleaning.MULTIPLIER_PATTERN
)

for value in [
    ".05% w/w",
    ".5mg",
    "60millionspores",
    "200mcg/spray",
    "300u/ml",
    "3500u/1g",
]:
    print(value, "->", cleaning.parse_strength(value))

Same strength regex: True
Same multiplier regex: True
.05% w/w -> {'strength_value': 0.05, 'strength_unit': '%', 'denominator_value': None, 'denominator_unit': None, 'strength_basis': 'w/w', 'strength_qualifier': None, 'parse_status': 'parsed'}
.5mg -> {'strength_value': 0.5, 'strength_unit': 'mg', 'denominator_value': None, 'denominator_unit': None, 'strength_basis': None, 'strength_qualifier': None, 'parse_status': 'parsed'}
60millionspores -> {'strength_value': 60.0, 'strength_unit': 'million', 'denominator_value': None, 'denominator_unit': None, 'strength_basis': None, 'strength_qualifier': 'spores', 'parse_status': 'parsed'}
200mcg/spray -> {'strength_value': 200.0, 'strength_unit': 'mcg', 'denominator_value': 1.0, 'denominator_unit': 'spray', 'strength_basis': None, 'strength_qualifier': None, 'parse_status': 'parsed'}
300u/ml -> {'strength_value': 300.0, 'strength_unit': 'units', 'denominator_value': 1.0, 'denominator_unit': 'ml', 'strength_basis': None, 'strength_qualifier': No

In [ ]:

clean_df["parsed_ingredients"] = clean_df.apply(cleaning.parse_medicine_compositions,axis=1)


ingredient_records = []

for _, row in clean_df.iterrows():
    for ingredient in row["parsed_ingredients"]:
        ingredient_records.append({
            "medicine_id": row["medicine_id"],
            **ingredient
        })

ingredients_df = pd.DataFrame(ingredient_records)

print("Total ingredient records:", len(ingredients_df))

print("\nParsing status:")
display(ingredients_df["parse_status"].value_counts())

print("\nParsing percentages:")
display((ingredients_df["parse_status"].value_counts(normalize=True) * 100).round(2))

remaining_unparsed = ingredients_df[ingredients_df["parse_status"] == "unparsed_strength"]

print("\nRemaining unparsed records:", len(remaining_unparsed))
print("Unique remaining formats:",remaining_unparsed["strength_raw"].nunique())

display(remaining_unparsed["strength_raw"].value_counts(dropna=False))

Total ingredient records: 366144

Parsing status:


parse_status
parsed               360677
missing_strength       5377
unparsed_strength        90
Name: count, dtype: int64


Parsing percentages:


parse_status
parsed               98.51
missing_strength      1.47
unparsed_strength     0.02
Name: proportion, dtype: float64


Remaining unparsed records: 90
Unique remaining formats: 46


strength_raw
5LF                    15
1000ccid50              9
25LF                    5
3miu                    5
5000ccid50              4
6Lac units              3
40D - Antigen units     3
8D - Antigen units      3
100000FFU               3
0.2CU                   2
2000PFU                 2
300mg I/ml              2
22.5LF                  1
6.5ccid50               1
1CU                     1
2% w/w/1gm              1
20LF                    1
2520LF                  1
57.5LF                  1
10000PFU                1
40000PFU                1
33.6MIU                 1
720ELISA                1
24Lac units             1
18miu                   1
5Lac units              1
10mspores               1
2mspores                1
5mg I/ml                1
10ppm                   1
1000PFU                 1
40D Antigen Unit        1
8D Antigen Unit         1
1000000ccid50           1
3000KIU                 1
5000FCC                 1
15LF                    1
1300CST                 1

In [36]:
import importlib
import src.cleaning as cleaning

importlib.reload(cleaning)

from src.cleaning import parse_strength

print(parse_strength("60millionspores"))

{'strength_value': 60.0, 'strength_unit': 'million', 'denominator_value': None, 'denominator_unit': None, 'strength_basis': None, 'strength_qualifier': 'spores', 'parse_status': 'parsed'}


In [37]:
# Regression tests for important strength formats

test_cases = {
    "500mg": "parsed",
    "2mg/5ml": "parsed",
    "30mg/ml": "parsed",
    "200mcg/spray": "parsed",
    "300u/ml": "parsed",
    "3500u/1g": "parsed",
    ".05% w/w": "parsed",
    "60millionspores": "parsed",
    "NA": "missing_strength",
    "5LF": "unparsed_strength",
    "1000ccid50": "unparsed_strength",
    "2000PFU": "unparsed_strength",
}

for strength, expected_status in test_cases.items():
    result = parse_strength(strength)

    actual_status = result["parse_status"]

    assert actual_status == expected_status, (
        f"{strength}: expected {expected_status}, got {actual_status}"
    )

print("All parser regression tests passed!")

All parser regression tests passed!


In [38]:
# Preserve original ingredient names
ingredients_df["ingredient_original"] = (
    ingredients_df["ingredient"].copy()
)

# Create normalized ingredient names
ingredients_df["ingredient_normalized"] = (
    ingredients_df["ingredient"]
    .map(cleaning.normalize_ingredient_name)
    .astype("string")
)

print("Original ingredient examples:")
display(
    ingredients_df[
        ["ingredient_original", "ingredient_normalized"]
    ].head(15)
)

Original ingredient examples:


,ingredient_original,ingredient_normalized
0,Amoxycillin,amoxycillin
1,Clavulanic Acid,clavulanic acid
2,Azithromycin,azithromycin
3,Ambroxol,ambroxol
4,Levosalbutamol,levosalbutamol
5,Fexofenadine,fexofenadine
6,Pheniramine,pheniramine
7,Montelukast,montelukast
8,Fexofenadine,fexofenadine
9,Amoxycillin,amoxycillin


In [39]:
print("Top 30 normalized ingredient names:")

display(
    ingredients_df["ingredient_normalized"]
    .value_counts()
    .head(30)
)

Top 30 normalized ingredient names:


ingredient_normalized
paracetamol                 17607
aceclofenac                  9126
domperidone                  9019
cefixime                     9000
ofloxacin                    8799
clavulanic acid              8559
amoxycillin                  8552
metformin                    7885
rabeprazole                  7350
levocetirizine               6207
ceftriaxone                  6185
cefpodoxime proxetil         6099
diclofenac                   6047
pantoprazole                 5909
azithromycin                 5558
glimepiride                  5308
montelukast                  4925
telmisartan                  4328
methylcobalamin              4047
chlorpheniramine maleate     3920
sulbactam                    3858
ambroxol                     3463
nimesulide                   3392
ornidazole                   3356
cefuroxime                   3267
amlodipine                   2833
mefenamic acid               2695
itraconazole                 2639
pregabalin                

In [40]:
# Search for possible spelling variations
search_terms = [
    "amox",
    "paracetamol",
    "cef",
    "azithromycin",
    "metformin",
    "diclofenac",
    "chlorpheniramine"
]

for term in search_terms:
    matches = ingredients_df[
        ingredients_df["ingredient_normalized"]
        .str.contains(term, case=False, na=False, regex=False)
    ]

    print(f"\n--- {term.upper()} ---")

    display(
        matches.groupby("ingredient_normalized")
        .agg(
            record_count=("ingredient_normalized", "size"),
            original_names=(
                "ingredient_original",
                lambda x: sorted(set(x.dropna().astype(str)))
            )
        )
        .sort_values("record_count", ascending=False)
    )


--- AMOX ---


,record_count,original_names
ingredient_normalized,,
amoxycillin,8552,[Amoxycillin]
tamoxifen,46,[Tamoxifen]
amoxapine,20,[Amoxapine]
pramoxine,1,[Pramoxine]



--- PARACETAMOL ---


,record_count,original_names
ingredient_normalized,,
paracetamol,17607,[Paracetamol]
paracetamol/acetaminophen,275,[Paracetamol/Acetaminophen]



--- CEF ---


,record_count,original_names
ingredient_normalized,,
cefixime,9000,[Cefixime]
ceftriaxone,6185,[Ceftriaxone]
cefpodoxime proxetil,6099,[Cefpodoxime Proxetil]
cefuroxime,3267,[Cefuroxime]
cefoperazone,1353,[Cefoperazone]
ceftazidime,697,[Ceftazidime]
cefalexin,636,[Cefalexin]
cefadroxil,612,[Cefadroxil]
cefotaxime,337,[Cefotaxime]


... (4 outputs left out)


--- DICLOFENAC ---


,record_count,original_names
ingredient_normalized,,
diclofenac,6047,[Diclofenac]
diclofenac diethylamine,141,[Diclofenac diethylamine]



--- CHLORPHENIRAMINE ---


,record_count,original_names
ingredient_normalized,,
chlorpheniramine maleate,3920,[Chlorpheniramine Maleate]
dexchlorpheniramine,66,[Dexchlorpheniramine]


In [ ]:
print("Unique original ingredient names:",ingredients_df["ingredient_original"].nunique())

print("Unique normalized ingredient names:",ingredients_df["ingredient_normalized"].nunique())

Unique original ingredient names: 1718
Unique normalized ingredient names: 1716


In [ ]:
normalization_changes = (ingredients_df.groupby("ingredient_normalized").agg(
        original_variants=("ingredient_original",lambda x: sorted(set(x.dropna().astype(str)))),
        variant_count=("ingredient_original", "nunique")
    ).reset_index())

normalization_changes = normalization_changes[normalization_changes["variant_count"] > 1]

display(normalization_changes)

,ingredient_normalized,original_variants,variant_count
534,eicosapentaenoic acid,"[Eicosapentaenoic Acid, Eicosapentaenoic acid]",2
1018,milk of magnesia,"[Milk Of Magnesia, Milk of Magnesia]",2


In [ ]:
paracetamol_alias = ingredients_df[ingredients_df["ingredient_normalized"]== "paracetamol/acetaminophen"]

display(
    paracetamol_alias[
        [
            "medicine_id",
            "ingredient_original",
            "strength_raw",
            "strength_value",
            "strength_unit",
            "parse_status"
        ]
    ].head(20)
)

print("Total records:", len(paracetamol_alias))

,medicine_id,ingredient_original,strength_raw,strength_value,strength_unit,parse_status
216,152,Paracetamol/Acetaminophen,325mg,325.0,mg,parsed
2622,1803,Paracetamol/Acetaminophen,NA,NaN,NaN,missing_strength
4972,3402,Paracetamol/Acetaminophen,500mg,500.0,mg,parsed
5320,3635,Paracetamol/Acetaminophen,325mg,325.0,mg,parsed
7608,5175,Paracetamol/Acetaminophen,NA,NaN,NaN,missing_strength
9340,6315,Paracetamol/Acetaminophen,325mg,325.0,mg,parsed
9569,6472,Paracetamol/Acetaminophen,325mg,325.0,mg,parsed
12084,8149,Paracetamol/Acetaminophen,500mg,500.0,mg,parsed
13838,9332,Paracetamol/Acetaminophen,325mg,325.0,mg,parsed
16392,11083,Paracetamol/Acetaminophen,325mg,325.0,mg,parsed


Total records: 275


In [45]:
import importlib
import src.cleaning as cleaning

importlib.reload(cleaning)

ingredients_df["ingredient_canonical"] = (
    ingredients_df["ingredient_normalized"]
    .map(cleaning.canonicalize_ingredient)
    .astype("string")
)

In [ ]:

alias_records = ingredients_df[ingredients_df["ingredient_normalized"]== "paracetamol/acetaminophen"]

print("Alias records:", len(alias_records))

print(
    "Canonical values:",
    alias_records["ingredient_canonical"].unique()
)

print(
    "Paracetamol records before canonicalization:",
    (ingredients_df["ingredient_normalized"] == "paracetamol").sum()
)

print(
    "Paracetamol records after canonicalization:",
    (ingredients_df["ingredient_canonical"] == "paracetamol").sum()
)

Alias records: 275
Canonical values: <ArrowStringArray>
['paracetamol']
Length: 1, dtype: string
Paracetamol records before canonicalization: 17607
Paracetamol records after canonicalization: 17882


In [49]:
parsed_df = ingredients_df[
    ingredients_df["parse_status"] == "parsed"
].copy()

print("Strength unit distribution:")
display(
    parsed_df["strength_unit"]
    .value_counts(dropna=False)
)

print("\nDenominator unit distribution:")
display(
    parsed_df["denominator_unit"]
    .value_counts(dropna=False)
)

Strength unit distribution:


strength_unit
mg         326747
%           21469
mcg          6301
g            2288
million      1622
IU           1620
ml            235
billion       221
AU            144
units          30
Name: count, dtype: int64


Denominator unit distribution:


denominator_unit
NaN      342361
ml        18275
g            38
spray         3
Name: count, dtype: int64

In [50]:
mass_units = ["mg", "mcg", "ug", "µg", "g", "gm"]

other_units = parsed_df[
    ~parsed_df["strength_unit"]
    .str.lower()
    .isin(mass_units)
]

print("Non-mass strength units:")
display(
    other_units[
        [
            "strength_raw",
            "strength_value",
            "strength_unit",
            "denominator_value",
            "denominator_unit",
            "strength_basis",
            "strength_qualifier"
        ]
    ].head(30)
)

Non-mass strength units:


,strength_raw,strength_value,strength_unit,denominator_value,denominator_unit,strength_basis,strength_qualifier
18,0.10% w/w,0.100,%,NaN,NaN,w/w,NaN
19,0.025% w/w,0.025,%,NaN,NaN,w/w,NaN
54,600000IU,600000.000,IU,NaN,NaN,NaN,NaN
74,1.5% w/w,1.500,%,NaN,NaN,w/w,NaN
75,0.3% w/w,0.300,%,NaN,NaN,w/w,NaN
104,25000IU,25000.000,IU,NaN,NaN,NaN,NaN
125,0.1% w/w,0.100,%,NaN,NaN,w/w,NaN
146,5% w/w,5.000,%,NaN,NaN,w/w,NaN
152,0.05% w/w,0.050,%,NaN,NaN,w/w,NaN
155,20% w/w,20.000,%,NaN,NaN,w/w,NaN


In [61]:
import importlib
import src.cleaning as cleaning

importlib.reload(cleaning)

<module 'src.cleaning' from 'd:\\Engsci_Study_materials\\Subject_Index\\Projects\\cheaper_medicine_finder\\src\\cleaning.py'>

In [62]:
strength_normalized = ingredients_df.apply(
    cleaning.normalize_strength_record,
    axis=1
)

strength_normalized_df = pd.DataFrame(
    strength_normalized.tolist(),
    index=ingredients_df.index
)

ingredients_df = pd.concat(
    [ingredients_df, strength_normalized_df],
    axis=1
)

print("Strength normalization completed!")

display(
    ingredients_df[
        [
            "strength_raw",
            "strength_value",
            "strength_unit",
            "strength_value_normalized",
            "strength_unit_normalized",
            "denominator_value_normalized",
            "denominator_unit_normalized",
            "strength_normalization_status"
        ]
    ].head(25)
)

Strength normalization completed!


,strength_raw,strength_value,strength_unit,strength_value_normalized,strength_value_normalized,strength_value_normalized,strength_unit_normalized,strength_unit_normalized,strength_unit_normalized,denominator_value_normalized,denominator_value_normalized,denominator_value_normalized,denominator_unit_normalized,denominator_unit_normalized,denominator_unit_normalized,strength_normalization_status,strength_normalization_status,strength_normalization_status
0,500mg,500.000,mg,500.000,500.000,500.000,mg,mg,mg,NaN,NaN,NaN,NaN,NaN,NaN,already_mg,already_mg,mass_converted
1,125mg,125.000,mg,125.000,125.000,125.000,mg,mg,mg,NaN,NaN,NaN,NaN,NaN,NaN,already_mg,already_mg,mass_converted
2,500mg,500.000,mg,500.000,500.000,500.000,mg,mg,mg,NaN,NaN,NaN,NaN,NaN,NaN,already_mg,already_mg,mass_converted
3,30mg/5ml,30.000,mg,30.000,30.000,30.000,mg,mg,mg,5.0,5.0,5.0,ml,ml,ml,already_mg,already_mg,mass_converted
4,1mg/5ml,1.000,mg,1.000,1.000,1.000,mg,mg,mg,5.0,5.0,5.0,ml,ml,ml,already_mg,already_mg,mass_converted
5,120mg,120.000,mg,120.000,120.000,120.000,mg,mg,mg,NaN,NaN,NaN,NaN,NaN,NaN,already_mg,already_mg,mass_converted
6,25mg,25.000,mg,25.000,25.000,25.000,mg,mg,mg,NaN,NaN,NaN,NaN,NaN,NaN,already_mg,already_mg,mass_converted
7,10mg,10.000,mg,10.000,10.000,10.000,mg,mg,mg,NaN,NaN,NaN,NaN,NaN,NaN,already_mg,already_mg,mass_converted
8,120mg,120.000,mg,120.000,120.000,120.000,mg,mg,mg,NaN,NaN,NaN,NaN,NaN,NaN,already_mg,already_mg,mass_converted
9,500mg,500.000,mg,500.000,500.000,500.000,mg,mg,mg,NaN,NaN,NaN,NaN,NaN,NaN,already_mg,already_mg,mass_converted


In [ ]:


test_data = [
    {
        "strength_raw": "500mg",
        "strength_value": 500,
        "strength_unit": "mg",
        "denominator_value": float("nan"),
        "denominator_unit": None,
        "parse_status": "parsed",
        "expected_value": 500,
        "expected_unit": "mg",
    },
    {
        "strength_raw": "1000mcg",
        "strength_value": 1000,
        "strength_unit": "mcg",
        "denominator_value": float("nan"),
        "denominator_unit": None,
        "parse_status": "parsed",
        "expected_value": 1,
        "expected_unit": "mg",
    },
    {
        "strength_raw": "1.4gm/5ml",
        "strength_value": 1.4,
        "strength_unit": "g",
        "denominator_value": 5,
        "denominator_unit": "ml",
        "parse_status": "parsed",
        "expected_value": 1400,
        "expected_unit": "mg",
    },
    {
        "strength_raw": "0.1% w/w",
        "strength_value": 0.1,
        "strength_unit": "%",
        "denominator_value": float("nan"),
        "denominator_unit": None,
        "parse_status": "parsed",
        "expected_value": 0.1,
        "expected_unit": "%",
    },
    {
        "strength_raw": "600000IU",
        "strength_value": 600000,
        "strength_unit": "IU",
        "denominator_value": float("nan"),
        "denominator_unit": None,
        "parse_status": "parsed",
        "expected_value": 600000,
        "expected_unit": "iu",
    },
]

for item in test_data:
    expected_value = item.pop("expected_value")
    expected_unit = item.pop("expected_unit")

    result = cleaning.normalize_strength_record(
        pd.Series(item)
    )

    assert abs(
        result["strength_value_normalized"] - expected_value
    ) < 1e-9

    assert result["strength_unit_normalized"] == expected_unit

print("All strength normalization tests passed!")

All strength normalization tests passed!


In [ ]:

ingredients_df = ingredients_df.loc[
    :,
    ~ingredients_df.columns.duplicated(keep="last")
].copy()

print("Duplicate columns remaining:",
      ingredients_df.columns.duplicated().sum())

print("Dataset shape:", ingredients_df.shape)

Duplicate columns remaining: 0
Dataset shape: (366144, 20)


In [ ]:
print("Normalization status:")

display(
    ingredients_df["strength_normalization_status"]
    .value_counts(dropna=False)
)

print("\nNormalized units:")

display(
    ingredients_df["strength_unit_normalized"]
    .value_counts(dropna=False)
)

print("\nSample conversions:")

display(
    ingredients_df[
        [
            "strength_raw",
            "strength_value",
            "strength_unit",
            "strength_value_normalized",
            "strength_unit_normalized",
            "denominator_value_normalized",
            "denominator_unit_normalized",
            "strength_normalization_status"
        ]
    ].head(25)
)

Normalization status:


strength_normalization_status
mass_converted        335336
unchanged_non_mass     25341
not_parsed              5467
Name: count, dtype: int64


Normalized units:


strength_unit_normalized
mg         335336
%           21469
NaN          5467
million      1622
iu           1620
ml            235
billion       221
au            144
units          30
Name: count, dtype: int64


Sample conversions:


,strength_raw,strength_value,strength_unit,strength_value_normalized,strength_unit_normalized,denominator_value_normalized,denominator_unit_normalized,strength_normalization_status
0,500mg,500.000,mg,500.000,mg,NaN,NaN,mass_converted
1,125mg,125.000,mg,125.000,mg,NaN,NaN,mass_converted
2,500mg,500.000,mg,500.000,mg,NaN,NaN,mass_converted
3,30mg/5ml,30.000,mg,30.000,mg,5.0,ml,mass_converted
4,1mg/5ml,1.000,mg,1.000,mg,5.0,ml,mass_converted
5,120mg,120.000,mg,120.000,mg,NaN,NaN,mass_converted
6,25mg,25.000,mg,25.000,mg,NaN,NaN,mass_converted
7,10mg,10.000,mg,10.000,mg,NaN,NaN,mass_converted
8,120mg,120.000,mg,120.000,mg,NaN,NaN,mass_converted
9,500mg,500.000,mg,500.000,mg,NaN,NaN,mass_converted


In [ ]:

print("Unique pack-size descriptions:",
      clean_df["pack_size"].nunique())


display(
    clean_df["pack_size"]
    .value_counts(dropna=False)
    .head(100)
)

Unique pack-size descriptions: 1929


pack_size
strip of 10 tablets               116540
vial of 1 Injection                17678
strip of 10 capsules               11931
strip of 6 tablets                  5718
bottle of 100 ml Syrup              5663
                                   ...  
tube of 30 gm Ointment               152
bottle of 100 ml Suspension          150
strip of 6 tablet dt                 149
bottle of 200 ml Oral Solution       144
bottle of 50 ml Lotion               142
Name: count, Length: 100, dtype: int64[pyarrow]

In [67]:
formulation_keywords = [
    "tablet", "capsule", "syrup", "suspension",
    "injection", "inhaler", "drops", "cream",
    "ointment", "gel", "powder", "solution",
    "spray", "lotion", "suppository",
    "sr", "er", "xr", "cr", "mr",
    "dr", "dt", "dispersible", "delayed release",
    "extended release", "sustained release"
]

pack_text = clean_df["pack_size"].fillna("").str.lower()

for keyword in formulation_keywords:
    count = pack_text.str.contains(
        keyword, regex=False
    ).sum()

    if count > 0:
        print(f"{keyword}: {count}")

tablet: 152574
capsule: 22112
syrup: 16747
suspension: 9251
injection: 32044
inhaler: 267
drops: 1423
cream: 5071
ointment: 1669
gel: 2593
powder: 1793
solution: 1922
spray: 366
lotion: 1022
suppository: 32
sr: 11243
er: 4440
xr: 190
cr: 5808
mr: 312
dr: 9252
dt: 4649


In [68]:
import re

pack_text = clean_df["pack_size"].fillna("").str.lower()

modifiers = [
    "sr", "er", "xr", "cr", "mr", "dr", "dt"
]

print("RELEASE / FORMULATION MODIFIERS")
for modifier in modifiers:
    pattern = rf"\b{modifier}\b"

    count = pack_text.str.contains(
        pattern, regex=True
    ).sum()

    print(f"{modifier.upper()}: {count}")

RELEASE / FORMULATION MODIFIERS
SR: 11243
ER: 2235
XR: 190
CR: 728
MR: 312
DR: 126
DT: 4649


In [69]:
modifier_pattern = (
    r"\b(sr|er|xr|cr|mr|dr|dt)\b"
)

modifier_examples = (
    clean_df.loc[
        pack_text.str.contains(
            modifier_pattern,
            regex=True
        ),
        "pack_size"
    ]
    .value_counts()
    .head(100)
)

display(modifier_examples)

C:\Users\HP\AppData\Local\Temp\ipykernel_8644\3807592579.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  pack_text.str.contains(


pack_size
strip of 10 capsule sr     5511
strip of 10 tablet sr      5116
strip of 10 tablet dt      4309
strip of 10 tablet er      1871
strip of 10 tablet cr       581
                           ... 
strip of 25 capsule cr        1
strip of 7 tablet cr          1
strip of 25 tablet cr         1
strip of 6 capsule sr         1
packet of 30 capsule mr       1
Name: count, Length: 100, dtype: int64[pyarrow]

In [71]:
import importlib
import src.cleaning as cleaning

importlib.reload(cleaning)

formulation_results = clean_df["pack_size"].apply(
    cleaning.extract_formulation
)

formulation_df = pd.DataFrame(
    formulation_results.tolist(),
    index=clean_df.index
)

clean_df = pd.concat(
    [clean_df, formulation_df],
    axis=1
)

In [74]:
unknown_forms = (
    clean_df.loc[
        clean_df["dosage_form_normalized"] == "unknown",
        "pack_size"
    ]
    .value_counts()
    .head(50)
)

display(unknown_forms)

pack_size
bottle of 100 ml Infusion             530
bottle of 100 ml Expectorant          260
box of 75 gm Soap                     234
packet of 75 gm Soap                  232
bottle of 100 ml Shampoo              116
packet of 1 Kit                       115
bottle of 100 ml Mouth Wash           114
bottle of 60 ml Shampoo                78
bottle of 500 ml Infusion              72
box of 1 Kit                           68
bottle of 50 ml Infusion               62
bottle of 60 ml Expectorant            53
bottle of 100 ml Liquid                50
packet of 5 respules                   44
bottle of 50 ml Shampoo                43
bottle of 300 ml Infusion              42
packet of 30 rotacaps                  40
bottle of 100 ml Gargle                34
packet of 1 gm Sachet                  33
bottle of 150 ml Mouth Wash            25
bottle of 75 ml Shampoo                21
bottle of 200 ml Infusion              20
sachet of 5 gm Granules                20
ampoule of 10 ml Infusio

In [ ]:
import importlib
import src.cleaning as cleaning

importlib.reload(cleaning)

formulation_results = clean_df["pack_size"].apply(
    cleaning.extract_formulation
)

formulation_df = pd.DataFrame(
    formulation_results.tolist(),
    index=clean_df.index
)


generated_columns = [
    "dosage_form_normalized",
    "formulation_modifier",
    "formulation_detail",
    "formulation_status"
]

clean_df = clean_df.drop(
    columns=generated_columns,
    errors="ignore"
)

clean_df = pd.concat(
    [clean_df, formulation_df],
    axis=1
)

In [76]:
print("DOSAGE FORM DISTRIBUTION")

display(
    clean_df["dosage_form_normalized"]
    .value_counts(dropna=False)
)

print("\nUNKNOWN DESCRIPTIONS")

display(
    clean_df.loc[
        clean_df["dosage_form_normalized"] == "unknown",
        "pack_size"
    ].value_counts().head(40)
)

print("\nTOTAL RECORDS:",
      len(clean_df))

print("IDENTIFIED:",
      (clean_df["dosage_form_normalized"] != "unknown").sum())

print("UNKNOWN:",
      (clean_df["dosage_form_normalized"] == "unknown").sum())

DOSAGE FORM DISTRIBUTION


dosage_form_normalized
tablet                  152574
injection                31607
capsule                  22112
syrup                    12903
suspension                9226
drops                     5282
cream                     5070
dry_syrup                 3844
gel                       1685
ointment                  1669
solution                  1158
lotion                    1022
infusion                   903
unknown                    608
oral_solution              583
soap                       526
prefilled_syringe          424
spray                      366
expectorant                338
powder                     327
shampoo                    271
inhaler                    267
kit                        249
mouthwash                  184
sachet                     132
granules                   127
nebulizer_unit             122
inhalation_capsule          99
suppository                 76
disintegrating_strip        64
gargle                      53
oral_liquid     


UNKNOWN DESCRIPTIONS


pack_size
bottle of 100 ml Liquid               50
bottle of 30 transcaps                20
strip of 3 Vaginal Suppository        15
bottle of 60 ml Liquid                13
bottle of 30 ml Liquid                11
bottle of 200 ml Liquid               11
packet of 1 Transdermal Patch         10
bottle of 100 ml Linctus               8
packet of 100 MDI Novocart             8
bottle of 60 ml Linctus                7
packet of 120 MDI Transhaler           7
box of 2 transdermal patches           7
packet of 200 MDI Transhaler           7
bottle of 500 ml Liquid                7
tube of 5 gm Oral Paste                7
tube of 5 gm Paste                     6
strip of 7 Vaginal Suppository         6
bottle of 15 ml Liquid                 6
strip of 10 tabcaps                    6
packet of 2 transdermal patches        6
packet of 3 Vaginal Suppository        6
packet of 30 redicaps                  6
packet of 6 pessaries                  6
bottle of 5 ml Ophthalmic Emulsion     6
bottle


TOTAL RECORDS: 253973
IDENTIFIED: 253365
UNKNOWN: 608


In [78]:
print("TOTAL RECORDS:", len(clean_df))

print("\nIDENTIFIED:")
print(
    (clean_df["dosage_form_normalized"] != "unknown").sum()
)

print("\nUNKNOWN:")
print(
    (clean_df["dosage_form_normalized"] == "unknown").sum()
)

print("\nREMAINING UNKNOWN DESCRIPTIONS:")
display(
    clean_df.loc[
        clean_df["dosage_form_normalized"] == "unknown",
        "pack_size"
    ].value_counts().head(40)
)

TOTAL RECORDS: 253973

IDENTIFIED:
253365

UNKNOWN:
608

REMAINING UNKNOWN DESCRIPTIONS:


pack_size
bottle of 100 ml Liquid               50
bottle of 30 transcaps                20
strip of 3 Vaginal Suppository        15
bottle of 60 ml Liquid                13
bottle of 30 ml Liquid                11
bottle of 200 ml Liquid               11
packet of 1 Transdermal Patch         10
bottle of 100 ml Linctus               8
packet of 100 MDI Novocart             8
bottle of 60 ml Linctus                7
packet of 120 MDI Transhaler           7
box of 2 transdermal patches           7
packet of 200 MDI Transhaler           7
bottle of 500 ml Liquid                7
tube of 5 gm Oral Paste                7
tube of 5 gm Paste                     6
strip of 7 Vaginal Suppository         6
bottle of 15 ml Liquid                 6
strip of 10 tabcaps                    6
packet of 2 transdermal patches        6
packet of 3 Vaginal Suppository        6
packet of 30 redicaps                  6
packet of 6 pessaries                  6
bottle of 5 ml Ophthalmic Emulsion     6
bottle

In [79]:
md_records = clean_df[
    clean_df["pack_size"].str.contains(
        r"\bmd\b",
        case=False,
        regex=True,
        na=False
    )
]

display(
    md_records[
        [
            "name",
            "pack_size",
            "composition1",
            "composition2"
        ]
    ].head(25)
)

,name,pack_size,composition1,composition2
359,Avomine 25mg Tablet MD,strip of 8 tablet md,Promethazine (25mg),<NA>
568,Altonil 5 Tablet MD,strip of 15 tablet md,Melatonin (5mg),<NA>
660,Anset 4mg Tablet MD,strip of 10 tablet md,Ondansetron (4mg),<NA>
787,Anxipam 0.5mg Tablet MD,strip of 10 tablet md,Clonazepam (0.5mg),<NA>
1886,Altonil Tablet MD,strip of 15 tablet md,Melatonin (3mg),<NA>
1983,Admenta 5mg Tablet MD,strip of 10 tablet md,Memantine (5mg),<NA>
2348,ADMENTA 10 MG TABLET MD,strip of 10 tablet md,Memantine (10mg),<NA>
2571,Arirazo 5mg Tablet MD,strip of 10 tablet md,Aripiprazole (5mg),<NA>
3348,Apiz 5mg Tablet MD,strip of 10 tablet md,Aripiprazole (5mg),<NA>
4079,Altonil 10 Tablet MD,strip of 15 tablet md,Melatonin (10mg),<NA>


In [80]:
from src.cleaning import DOSAGE_FORM_PATTERNS
import re

test_labels = [
    "bottle of 100 ml Liquid",
    "strip of 3 Vaginal Suppository",
    "packet of 1 Transdermal Patch",
    "bottle of 100 ml Linctus",
    "packet of 100 MDI Novocart",
    "tube of 5 gm Oral Paste",
    "bottle of 5 ml Ophthalmic Emulsion",
]

for label in test_labels:
    matches = [
        form
        for form, pattern in DOSAGE_FORM_PATTERNS
        if re.search(pattern, label, flags=re.IGNORECASE)
    ]

    print(label, "->", matches)

bottle of 100 ml Liquid -> []
strip of 3 Vaginal Suppository -> []
packet of 1 Transdermal Patch -> []
bottle of 100 ml Linctus -> []
packet of 100 MDI Novocart -> []
tube of 5 gm Oral Paste -> []
bottle of 5 ml Ophthalmic Emulsion -> []


In [ ]:
import importlib
import src.cleaning as cleaning

importlib.reload(cleaning)

formulation_results = clean_df["pack_size"].apply(
    cleaning.extract_formulation
)

formulation_df = pd.DataFrame(
    formulation_results.tolist(),
    index=clean_df.index
)

# Remove old generated columns before replacing them
generated_columns = [
    "dosage_form_normalized",
    "formulation_modifier",
    "formulation_detail",
    "formulation_status",
]

clean_df = clean_df.drop(
    columns=generated_columns,
    errors="ignore"
)

clean_df = pd.concat(
    [clean_df, formulation_df],
    axis=1
)

print(clean_df["dosage_form_normalized"].value_counts())
print(
    "Unknown:",
    (clean_df["dosage_form_normalized"] == "unknown").sum()
)

dosage_form_normalized
tablet                  152574
injection                31607
capsule                  22112
syrup                    12903
suspension                9226
drops                     5282
cream                     5070
dry_syrup                 3844
gel                       1685
ointment                  1669
solution                  1158
lotion                    1022
infusion                   903
oral_solution              583
soap                       526
prefilled_syringe          424
spray                      366
expectorant                338
unknown                    332
powder                     327
shampoo                    271
inhaler                    267
kit                        249
mouthwash                  184
sachet                     132
liquid                     128
granules                   127
nebulizer_unit             122
inhalation_capsule          99
suppository                 76
disintegrating_strip        64
gargle          

In [82]:
import src.cleaning as cleaning

test_labels = [
    "bottle of 100 ml Liquid",
    "strip of 3 Vaginal Suppository",
    "packet of 1 Transdermal Patch",
    "bottle of 100 ml Linctus",
    "packet of 100 MDI Novocart",
    "tube of 5 gm Oral Paste",
    "bottle of 5 ml Ophthalmic Emulsion",
]

for label in test_labels:
    matches = [
        form
        for form, pattern in cleaning.DOSAGE_FORM_PATTERNS
        if re.search(pattern, label, flags=re.IGNORECASE)
    ]
    print(label, "->", matches)

bottle of 100 ml Liquid -> ['liquid']
strip of 3 Vaginal Suppository -> []
packet of 1 Transdermal Patch -> ['transdermal_patch', 'transdermal_patch']
bottle of 100 ml Linctus -> ['linctus', 'linctus']
packet of 100 MDI Novocart -> ['inhaler_device']
tube of 5 gm Oral Paste -> ['oral_paste']
bottle of 5 ml Ophthalmic Emulsion -> ['ophthalmic_emulsion']


In [83]:
print(clean_df["formulation_modifier"].value_counts(dropna=False))

display(
    clean_df.loc[
        clean_df["formulation_modifier"] == "md",
        ["name", "pack_size", "dosage_form_normalized",
         "formulation_modifier", "formulation_detail"]
    ].head(15)
)

formulation_modifier
NaN    232654
sr      11243
dt       4649
er       2235
md       1836
cr        728
mr        312
xr        190
dr        126
Name: count, dtype: int64


,name,pack_size,dosage_form_normalized,formulation_modifier,formulation_detail
359,Avomine 25mg Tablet MD,strip of 8 tablet md,tablet,md,mouth_dissolving_candidate
568,Altonil 5 Tablet MD,strip of 15 tablet md,tablet,md,mouth_dissolving_candidate
660,Anset 4mg Tablet MD,strip of 10 tablet md,tablet,md,mouth_dissolving_candidate
787,Anxipam 0.5mg Tablet MD,strip of 10 tablet md,tablet,md,mouth_dissolving_candidate
1886,Altonil Tablet MD,strip of 15 tablet md,tablet,md,mouth_dissolving_candidate
1983,Admenta 5mg Tablet MD,strip of 10 tablet md,tablet,md,mouth_dissolving_candidate
2348,ADMENTA 10 MG TABLET MD,strip of 10 tablet md,tablet,md,mouth_dissolving_candidate
2571,Arirazo 5mg Tablet MD,strip of 10 tablet md,tablet,md,mouth_dissolving_candidate
3348,Apiz 5mg Tablet MD,strip of 10 tablet md,tablet,md,mouth_dissolving_candidate
4079,Altonil 10 Tablet MD,strip of 15 tablet md,tablet,md,mouth_dissolving_candidate


In [84]:
unknown_records = clean_df[
    clean_df["dosage_form_normalized"] == "unknown"
]

print("Unknown record count:", len(unknown_records))
print("Distinct unknown descriptions:",
      unknown_records["pack_size"].nunique())

display(
    unknown_records["pack_size"]
    .value_counts()
    .head(50)
)

Unknown record count: 332
Distinct unknown descriptions: 164


pack_size
bottle of 30 transcaps             20
strip of 3 Vaginal Suppository     15
tube of 5 gm Paste                  6
strip of 7 Vaginal Suppository      6
strip of 10 tabcaps                 6
packet of 3 Vaginal Suppository     6
packet of 30 redicaps               6
bottle of 2.5 ml Nail Lacquer       5
bottle of 170 ml Oral Emulsion      5
box of 10 bottles                   5
bottle of 30 ml Liniment            5
tube of 30 gm Jelly                 5
packet of 3 cartriges               5
bottle of 60 ml Serum               5
packet of 10 chewing gums           5
box of 10 pastilles                 4
box of 2 patches                    4
packet of 1 Patch                   4
tube of 50 gm Face Wash             3
packet of 5 transpules              3
strip of 10 caplets                 3
packet of 500 ml Scrub              3
box of 30 patches                   3
box of 3 patches                    3
bottle of 30 ml Serum               3
bottle of 60 gm Foam                3
pa

In [86]:
print("Total rows:", len(clean_df))

print(
    "Missing dosage form:",
    clean_df["dosage_form_normalized"].isna().sum()
)

print(
    "Unknown dosage form:",
    clean_df["dosage_form_normalized"].eq("unknown").sum()
)

print(
    "Identified dosage form:",
    clean_df["dosage_form_normalized"].notna()
    .mul(clean_df["dosage_form_normalized"].ne("unknown"))
    .sum()
)

print(
    "Total accounted:",
    clean_df["dosage_form_normalized"].notna().sum()
)

Total rows: 253973
Missing dosage form: 0
Unknown dosage form: 332
Identified dosage form: 253641
Total accounted: 253973


In [87]:
unknown_records = clean_df[
    clean_df["dosage_form_normalized"] == "unknown"
]

print("Unknown rows:", len(unknown_records))
print(
    "Unique descriptions:",
    unknown_records["pack_size"].nunique()
)

display(
    unknown_records["pack_size"]
    .value_counts()
    .head(40)
)

Unknown rows: 332
Unique descriptions: 164


pack_size
bottle of 30 transcaps             20
strip of 3 Vaginal Suppository     15
tube of 5 gm Paste                  6
strip of 7 Vaginal Suppository      6
strip of 10 tabcaps                 6
packet of 3 Vaginal Suppository     6
packet of 30 redicaps               6
bottle of 2.5 ml Nail Lacquer       5
bottle of 170 ml Oral Emulsion      5
box of 10 bottles                   5
bottle of 30 ml Liniment            5
tube of 30 gm Jelly                 5
packet of 3 cartriges               5
bottle of 60 ml Serum               5
packet of 10 chewing gums           5
box of 10 pastilles                 4
box of 2 patches                    4
packet of 1 Patch                   4
tube of 50 gm Face Wash             3
packet of 5 transpules              3
strip of 10 caplets                 3
packet of 500 ml Scrub              3
box of 30 patches                   3
box of 3 patches                    3
bottle of 30 ml Serum               3
bottle of 60 gm Foam                3
pa

In [89]:
import re
import importlib
import src.cleaning as cleaning

importlib.reload(cleaning)

test_labels = [
    "strip of 3 Vaginal Suppository",
    "tube of 5 gm Paste",
    "bottle of 2.5 ml Nail Lacquer",
    "bottle of 170 ml Oral Emulsion",
    "bottle of 30 ml Liniment",
    "tube of 30 gm Jelly",
    "packet of 10 chewing gums",
    "box of 10 pastilles",
    "box of 2 patches",
    "tube of 50 gm Face Wash",
]

for label in test_labels:
    matches = [
        form
        for form, pattern in cleaning.DOSAGE_FORM_PATTERNS
        if re.search(pattern, label, flags=re.IGNORECASE)
    ]

    print(label, "->", matches)

strip of 3 Vaginal Suppository -> []
tube of 5 gm Paste -> []
bottle of 2.5 ml Nail Lacquer -> ['nail_lacquer', 'nail_lacquer']
bottle of 170 ml Oral Emulsion -> ['oral_emulsion', 'oral_emulsion']
bottle of 30 ml Liniment -> ['liniment', 'liniment']
tube of 30 gm Jelly -> ['jelly', 'jelly']
packet of 10 chewing gums -> ['chewing_gum', 'chewing_gum']
box of 10 pastilles -> ['pastille', 'pastille']
box of 2 patches -> ['patch', 'patch']
tube of 50 gm Face Wash -> ['face_wash', 'face_wash']


In [90]:
formulation_results = clean_df["pack_size"].apply(
    cleaning.extract_formulation
)

formulation_df = pd.DataFrame(
    formulation_results.tolist(),
    index=clean_df.index
)

generated_columns = [
    "dosage_form_normalized",
    "formulation_modifier",
    "formulation_detail",
    "formulation_status",
]

clean_df = clean_df.drop(
    columns=generated_columns,
    errors="ignore"
)

clean_df = pd.concat(
    [clean_df, formulation_df],
    axis=1
)

print("Total:", len(clean_df))
print(
    "Unknown:",
    clean_df["dosage_form_normalized"].eq("unknown").sum()
)
print(
    "Missing:",
    clean_df["dosage_form_normalized"].isna().sum()
)

Total: 253973
Unknown: 217
Missing: 0


In [91]:
import importlib
import src.cleaning as cleaning
importlib.reload(cleaning)

test_labels = [
    "strip of 3 Vaginal Suppository",
    "strip of 7 Vaginal Suppository",
    "tube of 5 gm Paste",
    "tube of 5 gm Oral Paste",
]

for label in test_labels:
    matches = [
        form
        for form, pattern in cleaning.DOSAGE_FORM_PATTERNS
        if re.search(pattern, label, flags=re.IGNORECASE)
    ]
    print(label, "->", matches)

strip of 3 Vaginal Suppository -> ['vaginal_suppository']
strip of 7 Vaginal Suppository -> ['vaginal_suppository']
tube of 5 gm Paste -> ['paste']
tube of 5 gm Oral Paste -> ['paste', 'oral_paste']


In [92]:
print("Total:", len(clean_df))
print("Unknown:", clean_df["dosage_form_normalized"].eq("unknown").sum())
print("Missing:", clean_df["dosage_form_normalized"].isna().sum())

display(
    clean_df.loc[
        clean_df["dosage_form_normalized"] == "unknown",
        "pack_size"
    ].value_counts().head(30)
)

Total: 253973
Unknown: 217
Missing: 0


pack_size
bottle of 30 transcaps             20
strip of 3 Vaginal Suppository     15
tube of 5 gm Paste                  6
strip of 7 Vaginal Suppository      6
strip of 10 tabcaps                 6
packet of 3 Vaginal Suppository     6
packet of 30 redicaps               6
box of 10 bottles                   5
packet of 3 cartriges               5
packet of 1 Patch                   4
packet of 5 transpules              3
strip of 10 caplets                 3
packet of 30 Nexcaps                3
packet of 1 Unit                    3
packet of 30 Oxycaps                3
packet of 28 rheocap                3
packet of 2.5 ml Transpules         3
packet of 1 Pouch                   3
bottle of 15 transcaps              3
bottle of 30 instacaps              2
packet of 30 instacaps              2
box of 4 bottles                    2
box of 30 gummies                   2
bottle of 10 ml Particles           2
bottle of 100 ml Body Wash          2
packet of 1 Pen                     2
pa

In [93]:
import importlib
import src.cleaning as cleaning

importlib.reload(cleaning)

formulation_results = clean_df["pack_size"].apply(
    cleaning.extract_formulation
)

formulation_df = pd.DataFrame(
    formulation_results.tolist(),
    index=clean_df.index
)

generated_columns = [
    "dosage_form_normalized",
    "formulation_modifier",
    "formulation_detail",
    "formulation_status",
]

clean_df = clean_df.drop(
    columns=generated_columns,
    errors="ignore"
)

clean_df = pd.concat(
    [clean_df, formulation_df],
    axis=1
)

In [96]:
unknown_pack_sizes = (
    clean_df.loc[
        clean_df["dosage_form_normalized"] == "unknown",
        "pack_size"
    ]
    .value_counts()
)

print("Distinct unknown descriptions:", len(unknown_pack_sizes))
display(unknown_pack_sizes.head(40))

Distinct unknown descriptions: 99


pack_size
bottle of 30 transcaps             20
strip of 10 tabcaps                 6
packet of 30 redicaps               6
box of 10 bottles                   5
packet of 3 cartriges               5
packet of 1 Patch                   4
packet of 5 transpules              3
strip of 10 caplets                 3
packet of 30 Nexcaps                3
packet of 1 Unit                    3
packet of 30 Oxycaps                3
packet of 28 rheocap                3
packet of 2.5 ml Transpules         3
packet of 1 Pouch                   3
bottle of 15 transcaps              3
bottle of 30 instacaps              2
packet of 30 instacaps              2
box of 4 bottles                    2
box of 30 gummies                   2
bottle of 10 ml Particles           2
bottle of 100 ml Body Wash          2
packet of 1 Pen                     2
packet of 1 ml Transpules           2
box of 2 bottles                    2
bottle of 5 ml Particles            2
packet of 8 diskettes               2
st

In [97]:
unknown_records = clean_df.loc[
    clean_df["dosage_form_normalized"] == "unknown",
    [
        "medicine_id",
        "name",
        "manufacturer",
        "pack_size",
        "composition1",
        "composition2",
    ]
].copy()

print("Unknown medicine records:", len(unknown_records))

display(
    unknown_records.sort_values("pack_size").head(100)
)

Unknown medicine records: 177


,medicine_id,name,manufacturer,pack_size,composition1,composition2
2387,2388,Amcrylate Bio-Adhesive,Concord Drugs Ltd,ampoule of 0.25 ml Bio-Adhesive,Iso Amyl 2-Cynoacrylate (NA),<NA>
7361,7362,Amcrylate Bio-Adhesive,Concord Drugs Ltd,ampoule of 0.5 ml Bio-Adhesive,Iso Amyl 2-Cynoacrylate (NA),<NA>
86375,86376,Fluoritop SR Dental Varnish 1.5 ml,Icpa Health Products Ltd,bottle of 1 Paint,Sodium Fluoride (50mg),<NA>
143613,143614,Meslataj 1gm Pellets,Taj Pharma India Ltd,bottle of 1 Pellets,Mesalazine (1gm),<NA>
148604,148605,Meslataj 500mg Pellets,Taj Pharma India Ltd,bottle of 1 Pellets,Mesalazine (500mg),<NA>
...,...,...,...,...,...,...
203521,203522,Scc 4 LW Plus Pouch,Maneesh Pharmaceuticals Ltd,packet of 1 Pouch,Isoniazid (NA),Rifampicin (NA)
124694,124695,Levodiva Device,Bharat Serums & Vaccines Ltd,packet of 1 Unit,Levonorgestrel (intrauterine delivery system) ...,<NA>
134049,134050,Mirena Intrauterine Delivery System,Bayer Zydus Pharma Pvt Ltd,packet of 1 Unit,Levonorgestrel (intrauterine delivery system) ...,<NA>
153078,153079,Nuva Ring Device,Organon (India) Ltd,packet of 1 Unit,Ethinyl Estradiol (0.015mg),Etonogestrel (0.120mg)


In [98]:
unknown_descriptions = (
    unknown_records["pack_size"]
    .value_counts()
    .rename_axis("pack_size")
    .reset_index(name="count")
)

display(unknown_descriptions)

,pack_size,count
0,bottle of 30 transcaps,20
1,strip of 10 tabcaps,6
2,packet of 30 redicaps,6
3,box of 10 bottles,5
4,packet of 3 cartriges,5
...,...,...
94,packet of 100 opticops,1
95,bottle of 20 ml Paint,1
96,combo pack of 2 bottles,1
97,bottle of 10 ml Paint,1


In [101]:
import importlib
import src.cleaning as cleaning

importlib.reload(cleaning)

formulation_results = clean_df["pack_size"].apply(
    cleaning.extract_formulation
)

formulation_df = pd.DataFrame(
    formulation_results.tolist(),
    index=clean_df.index
)

generated_columns = [
    "dosage_form_normalized",
    "formulation_modifier",
    "formulation_detail",
    "formulation_status",
]

clean_df = clean_df.drop(
    columns=generated_columns,
    errors="ignore"
)

clean_df = pd.concat(
    [clean_df, formulation_df],
    axis=1
)

In [102]:
print("Total:", len(clean_df))
print(
    "Unknown:",
    clean_df["dosage_form_normalized"].eq("unknown").sum()
)
print(
    "Missing:",
    clean_df["dosage_form_normalized"].isna().sum()
)

display(clean_df["dosage_form_normalized"].value_counts())

Total: 253973
Unknown: 143
Missing: 0


dosage_form_normalized
tablet                 152574
injection               31607
capsule                 22112
syrup                   12903
suspension               9226
                        ...  
cleanser                    1
vaginal_wash                1
intrauterine_device         1
aerosol                     1
tonic                       1
Name: count, Length: 69, dtype: int64

In [103]:
check_names = [
    "Amcrylate Bio-Adhesive",
    "Fluoritop SR Dental Varnish 1.5 ml",
    "Meslataj 1gm Pellets",
    "Mirena Intrauterine Delivery System",
    "BETADINE 10% PAINT",
    "Viloral Mouth Paint",
]

display(
    clean_df[
        clean_df["name"].isin(check_names)
    ][["name", "pack_size", "dosage_form_normalized"]]
)

,name,pack_size,dosage_form_normalized
2387,Amcrylate Bio-Adhesive,ampoule of 0.25 ml Bio-Adhesive,bio_adhesive
7361,Amcrylate Bio-Adhesive,ampoule of 0.5 ml Bio-Adhesive,bio_adhesive
27654,BETADINE 10% PAINT,bottle of 50 ml Paint,paint
86375,Fluoritop SR Dental Varnish 1.5 ml,bottle of 1 Paint,paint
134049,Mirena Intrauterine Delivery System,packet of 1 Unit,unknown
143613,Meslataj 1gm Pellets,bottle of 1 Pellets,pellets
236765,Viloral Mouth Paint,bottle of 20 ml Paint,paint


In [ ]:

ingredient_counts = (
    ingredients_df
    .groupby("medicine_id")
    .size()
)

print("Unique medicines in clean_df:", clean_df["medicine_id"].nunique())
print("Medicines with ingredient records:", len(ingredient_counts))

print("\nIngredient count distribution:")
display(ingredient_counts.value_counts().sort_index())

print("\nMedicines with more than 2 ingredient records:")
display(
    ingredient_counts[ingredient_counts > 2].head(20)
)

Unique medicines in clean_df: 253973
Medicines with ingredient records: 253973

Ingredient count distribution:


1    141802
2    112171
Name: count, dtype: int64


Medicines with more than 2 ingredient records:


Series([], dtype: int64)

In [105]:
print("Total ingredient records:", len(ingredients_df))

print("\nParsing status:")
display(ingredients_df["parse_status"].value_counts())

print("\nNormalization status:")
display(
    ingredients_df["strength_normalization_status"].value_counts()
)

print("\nMissing canonical ingredient names:")
print(
    ingredients_df["ingredient_canonical"].isna().sum()
)

print("\nMissing normalized strength values:")
print(
    ingredients_df["strength_value_normalized"].isna().sum()
)

Total ingredient records: 366144

Parsing status:


parse_status
parsed               360677
missing_strength       5377
unparsed_strength        90
Name: count, dtype: int64


Normalization status:


strength_normalization_status
mass_converted        335336
unchanged_non_mass     25341
not_parsed              5467
Name: count, dtype: int64


Missing canonical ingredient names:
0

Missing normalized strength values:
5467


In [106]:
duplicate_ingredient_records = ingredients_df.duplicated(
    subset=["medicine_id", "ingredient_canonical"],
    keep=False
)

print(
    "Duplicate medicine-ingredient combinations:",
    duplicate_ingredient_records.sum()
)

display(
    ingredients_df.loc[
        duplicate_ingredient_records,
        [
            "medicine_id",
            "ingredient",
            "ingredient_canonical",
            "strength_raw",
            "parse_status",
        ]
    ].head(20)
)

Duplicate medicine-ingredient combinations: 82


,medicine_id,ingredient,ingredient_canonical,strength_raw,parse_status
182,128,Paracetamol,paracetamol,300mg,parsed
183,128,Paracetamol,paracetamol,700mg,parsed
612,421,Aprepitant,aprepitant,125mg,parsed
613,421,Aprepitant,aprepitant,80mg,parsed
2436,1674,Paracetamol,paracetamol,300mg,parsed
2437,1674,Paracetamol,paracetamol,700mg,parsed
3393,2335,Aprepitant,aprepitant,125mg,parsed
3394,2335,Aprepitant,aprepitant,80mg,parsed
3527,2429,Aprepitant,aprepitant,125mg,parsed
3528,2429,Aprepitant,aprepitant,80mg,parsed


In [107]:
duplicate_groups = (
    ingredients_df
    .groupby(
        ["medicine_id", "ingredient_canonical"],
        dropna=False
    )
    .filter(lambda group: len(group) > 1)
    .sort_values(["medicine_id", "ingredient_canonical"])
)

display(
    duplicate_groups[
        [
            "medicine_id",
            "raw_composition",
            "ingredient",
            "ingredient_canonical",
            "strength_raw",
            "strength_value_normalized",
            "strength_unit_normalized",
            "source_column",
            "parse_status",
        ]
    ].head(50)
)

,medicine_id,raw_composition,ingredient,ingredient_canonical,strength_raw,strength_value_normalized,strength_unit_normalized,source_column,parse_status
182,128,Paracetamol (300mg),Paracetamol,paracetamol,300mg,300.0,mg,composition1,parsed
183,128,Paracetamol (700mg),Paracetamol,paracetamol,700mg,700.0,mg,composition2,parsed
612,421,Aprepitant (125mg),Aprepitant,aprepitant,125mg,125.0,mg,composition1,parsed
613,421,Aprepitant (80mg),Aprepitant,aprepitant,80mg,80.0,mg,composition2,parsed
2436,1674,Paracetamol (300mg),Paracetamol,paracetamol,300mg,300.0,mg,composition1,parsed
2437,1674,Paracetamol (700mg),Paracetamol,paracetamol,700mg,700.0,mg,composition2,parsed
3393,2335,Aprepitant (125mg),Aprepitant,aprepitant,125mg,125.0,mg,composition1,parsed
3394,2335,Aprepitant (80mg),Aprepitant,aprepitant,80mg,80.0,mg,composition2,parsed
3527,2429,Aprepitant (125mg),Aprepitant,aprepitant,125mg,125.0,mg,composition1,parsed
3528,2429,Aprepitant (80mg),Aprepitant,aprepitant,80mg,80.0,mg,composition2,parsed


In [109]:
component_columns = [
    "medicine_id",
    "ingredient_canonical",
    "strength_value_normalized",
    "strength_unit_normalized",
    "denominator_value_normalized",
    "denominator_unit_normalized",
    "strength_basis",
    "strength_qualifier",
]

repeated_components = (
    ingredients_df
    .groupby(
        component_columns,
        dropna=False
    )
    .size()
    .reset_index(name="component_count")
)

repeated_components = repeated_components[
    repeated_components["component_count"] > 1
]

print("Exact repeated component groups:", len(repeated_components))

display(repeated_components.head(30))

Exact repeated component groups: 3


,medicine_id,ingredient_canonical,strength_value_normalized,strength_unit_normalized,denominator_value_normalized,denominator_unit_normalized,strength_basis,strength_qualifier,component_count
9635,6514,aprepitant,125.0,mg,NaN,NaN,NaN,NaN,2
121909,84451,estradiol,1.0,mg,NaN,NaN,NaN,NaN,2
121941,84474,estradiol,2.0,mg,NaN,NaN,NaN,NaN,2


In [110]:
repeated_medicine_ids = (
    repeated_components["medicine_id"]
    .unique()
)

display(
    clean_df.loc[
        clean_df["medicine_id"].isin(repeated_medicine_ids),
        [
            "medicine_id",
            "name",
            "pack_size",
            "composition1",
            "composition2",
            "dosage_form_normalized",
        ]
    ].sort_values("medicine_id")
)

,medicine_id,name,pack_size,composition1,composition2,dosage_form_normalized
6513,6514,Aprelief Kit,strip of 1 Kit,Aprepitant (125mg),Aprepitant (125mg),kit
84450,84451,Femoston Tablet,strip of 28 tablets,Estradiol (1mg),Estradiol (1mg),tablet
84473,84474,Femoston 2/10 Tablet,box of 28 tablet combikit,Estradiol (2mg),Estradiol (2mg),tablet


In [ ]:
COMPONENT_IDENTITY_COLUMNS = [
    "medicine_id",
    "ingredient_canonical",
    "strength_value_normalized",
    "strength_unit_normalized",
    "denominator_value_normalized",
    "denominator_unit_normalized",
    "strength_basis",
    "strength_qualifier",
]

ingredients_df["exact_duplicate_component"] = (
    ingredients_df.duplicated(
        subset=COMPONENT_IDENTITY_COLUMNS,
        keep=False
    )
)


ingredients_df["medicine_has_duplicate_component"] = (
    ingredients_df
    .groupby("medicine_id")["exact_duplicate_component"]
    .transform("any")
)

In [113]:
duplicate_medicine_report = (
    ingredients_df.loc[
        ingredients_df["medicine_has_duplicate_component"],
        ["medicine_id"]
    ]
    .drop_duplicates()
    .merge(
        clean_df[
            [
                "medicine_id",
                "name",
                "composition1",
                "composition2",
            ]
        ],
        on="medicine_id",
        how="left"
    )
)

print("Medicines flagged:",
      len(duplicate_medicine_report))

display(duplicate_medicine_report)

Medicines flagged: 3


,medicine_id,name,composition1,composition2
0,6514,Aprelief Kit,Aprepitant (125mg),Aprepitant (125mg)
1,84451,Femoston Tablet,Estradiol (1mg),Estradiol (1mg)
2,84474,Femoston 2/10 Tablet,Estradiol (2mg),Estradiol (2mg)


In [116]:
import json
import pandas as pd


def normalize_signature_value(value):
    if pd.isna(value):
        return None

    if isinstance(value, (int, float)):
        return float(value)

    return str(value).strip().lower()


def create_component_signature(row):
    component = {
        "ingredient": row["ingredient_canonical"],
        "strength_value": normalize_signature_value(
            row["strength_value_normalized"]
        ),
        "strength_unit": normalize_signature_value(
            row["strength_unit_normalized"]
        ),
        "denominator_value": normalize_signature_value(
            row["denominator_value_normalized"]
        ),
        "denominator_unit": normalize_signature_value(
            row["denominator_unit_normalized"]
        ),
        "basis": normalize_signature_value(
            row["strength_basis"]
        ),
        "qualifier": normalize_signature_value(
            row["strength_qualifier"]
        ),
        "parse_status": normalize_signature_value(
            row["parse_status"]
        ),
    }

    return component


ingredients_df["component_signature"] = (
    ingredients_df.apply(
        create_component_signature,
        axis=1
    )
)

In [117]:
def create_medicine_signature(group):

    components = group["component_signature"].tolist()

    # Sort components so ingredient order does not affect identity
    components = sorted(
        components,
        key=lambda item: json.dumps(
            item,
            sort_keys=True
        )
    )

    return json.dumps(
        components,
        sort_keys=True,
        separators=(",", ":")
    )


medicine_signatures = (
    ingredients_df
    .groupby("medicine_id")
    .apply(create_medicine_signature)
    .reset_index(name="canonical_ingredient_signature")
)

print("Medicines with signatures:",
      len(medicine_signatures))

display(medicine_signatures.head(10))

Medicines with signatures: 253973


,medicine_id,canonical_ingredient_signature
0,1,"[{""basis"":null,""denominator_unit"":null,""denomi..."
1,2,"[{""basis"":null,""denominator_unit"":null,""denomi..."
2,3,"[{""basis"":null,""denominator_unit"":""ml"",""denomi..."
3,4,"[{""basis"":null,""denominator_unit"":null,""denomi..."
4,5,"[{""basis"":null,""denominator_unit"":null,""denomi..."
5,6,"[{""basis"":null,""denominator_unit"":null,""denomi..."
6,7,"[{""basis"":null,""denominator_unit"":null,""denomi..."
7,8,"[{""basis"":null,""denominator_unit"":null,""denomi..."
8,9,"[{""basis"":null,""denominator_unit"":null,""denomi..."
9,10,"[{""basis"":null,""denominator_unit"":null,""denomi..."


In [120]:
# Build medicine quality table
medicine_quality = (
    ingredients_df
    .groupby("medicine_id")
    .agg(
        parsed_ingredient_count=("ingredient_canonical", "size"),

        has_duplicate_component=(
            "exact_duplicate_component",
            "any"
        ),

        has_unparsed_strength=(
            "parse_status",
            lambda x: x.ne("parsed").any()
        ),

        has_unparsed_normalization=(
            "strength_normalization_status",
            lambda x: x.eq("not_parsed").any()
        )
    )
    .reset_index()
)

# Build medicine identity table
medicine_identity_df = (
    clean_df
    .merge(
        medicine_signatures,
        on="medicine_id",
        how="left"
    )
    .merge(
        medicine_quality,
        on="medicine_id",
        how="left"
    )
)

print("Medicine identity table:", medicine_identity_df.shape)

# Display results
display(
    medicine_identity_df[
        [
            "medicine_id",
            "name",
            "canonical_ingredient_signature",
            "ingredient_count",
            "parsed_ingredient_count",
            "has_duplicate_component",
            "has_unparsed_strength",
        ]
    ].head(10)
)

Medicine identity table: (253973, 22)


,medicine_id,name,canonical_ingredient_signature,ingredient_count,parsed_ingredient_count,has_duplicate_component,has_unparsed_strength
0,1,Augmentin 625 Duo Tablet,"[{""basis"":null,""denominator_unit"":null,""denomi...",2,2,False,False
1,2,Azithral 500 Tablet,"[{""basis"":null,""denominator_unit"":null,""denomi...",1,1,False,False
2,3,Ascoril LS Syrup,"[{""basis"":null,""denominator_unit"":""ml"",""denomi...",2,2,False,False
3,4,Allegra 120mg Tablet,"[{""basis"":null,""denominator_unit"":null,""denomi...",1,1,False,False
4,5,Avil 25 Tablet,"[{""basis"":null,""denominator_unit"":null,""denomi...",1,1,False,False
5,6,Allegra-M Tablet,"[{""basis"":null,""denominator_unit"":null,""denomi...",2,2,False,False
6,7,Amoxyclav 625 Tablet,"[{""basis"":null,""denominator_unit"":null,""denomi...",2,2,False,False
7,8,Azee 500 Tablet,"[{""basis"":null,""denominator_unit"":null,""denomi...",1,1,False,False
8,9,Atarax 25mg Tablet,"[{""basis"":null,""denominator_unit"":null,""denomi...",1,1,False,False
9,10,Ascoril D Plus Syrup Sugar Free,"[{""basis"":null,""denominator_unit"":null,""denomi...",2,2,False,False


In [121]:
print("Total medicines:", len(medicine_identity_df))

print(
    "\nMedicines with duplicate components:",
    medicine_identity_df["has_duplicate_component"].sum()
)

print(
    "\nMedicines with unparsed strengths:",
    medicine_identity_df["has_unparsed_strength"].sum()
)

print(
    "\nMedicines with missing canonical signatures:",
    medicine_identity_df["canonical_ingredient_signature"].isna().sum()
)

print("\nIngredient count distribution:")

display(
    medicine_identity_df["ingredient_count"]
    .value_counts()
    .sort_index()
)

print("\nIngredient parsing quality:")

display(
    medicine_identity_df["has_unparsed_strength"]
    .value_counts(dropna=False)
)

Total medicines: 253973

Medicines with duplicate components: 3

Medicines with unparsed strengths: 3872

Medicines with missing canonical signatures: 0

Ingredient count distribution:


ingredient_count
1    141802
2    112171
Name: count, dtype: int64


Ingredient parsing quality:


has_unparsed_strength
False    250101
True       3872
Name: count, dtype: int64

In [122]:
sample_names = [
    "Augmentin 625 Duo Tablet",
    "Ascoril LS Syrup",
    "Allegra-M Tablet",
]

for _, row in medicine_identity_df[
    medicine_identity_df["name"].isin(sample_names)
].iterrows():

    print("\nMedicine:", row["name"])
    print(row["canonical_ingredient_signature"])


Medicine: Augmentin 625 Duo Tablet
[{"basis":null,"denominator_unit":null,"denominator_value":null,"ingredient":"amoxycillin","parse_status":"parsed","qualifier":null,"strength_unit":"mg","strength_value":500.0},{"basis":null,"denominator_unit":null,"denominator_value":null,"ingredient":"clavulanic acid","parse_status":"parsed","qualifier":null,"strength_unit":"mg","strength_value":125.0}]

Medicine: Ascoril LS Syrup
[{"basis":null,"denominator_unit":"ml","denominator_value":5.0,"ingredient":"ambroxol","parse_status":"parsed","qualifier":null,"strength_unit":"mg","strength_value":30.0},{"basis":null,"denominator_unit":"ml","denominator_value":5.0,"ingredient":"levosalbutamol","parse_status":"parsed","qualifier":null,"strength_unit":"mg","strength_value":1.0}]

Medicine: Allegra-M Tablet
[{"basis":null,"denominator_unit":null,"denominator_value":null,"ingredient":"fexofenadine","parse_status":"parsed","qualifier":null,"strength_unit":"mg","strength_value":120.0},{"basis":null,"denomina

In [123]:
print("Dosage form distribution:")

display(
    medicine_identity_df["dosage_form_normalized"]
    .value_counts(dropna=False)
)

print(
    "\nUnknown dosage forms:",
    medicine_identity_df["dosage_form_normalized"]
    .eq("unknown")
    .sum()
)

print(
    "\nMissing dosage forms:",
    medicine_identity_df["dosage_form_normalized"]
    .isna()
    .sum()
)

Dosage form distribution:


dosage_form_normalized
tablet                 152574
injection               31607
capsule                 22112
syrup                   12903
suspension               9226
                        ...  
cleanser                    1
vaginal_wash                1
intrauterine_device         1
aerosol                     1
tonic                       1
Name: count, Length: 69, dtype: int64


Unknown dosage forms: 143

Missing dosage forms: 0


In [124]:
print("Release modifier distribution:")

display(
    medicine_identity_df["formulation_modifier"]
    .value_counts(dropna=False)
)

Release modifier distribution:


formulation_modifier
NaN    232654
sr      11243
dt       4649
er       2235
md       1836
cr        728
mr        312
xr        190
dr        126
Name: count, dtype: int64

In [125]:
modifier_records = medicine_identity_df.loc[
    medicine_identity_df["formulation_modifier"].notna(),
    [
        "medicine_id",
        "name",
        "pack_size",
        "dosage_form_normalized",
        "formulation_modifier",
    ]
].copy()

display(modifier_records.head(30))

print(
    "\nTotal medicines with a release modifier:",
    len(modifier_records)
)

,medicine_id,name,pack_size,dosage_form_normalized,formulation_modifier
22,23,Altraday Capsule SR,strip of 10 capsule sr,capsule,sr
73,74,AB Phylline SR 200 Tablet,strip of 10 tablet sr,tablet,sr
89,90,Acivir 400 DT Tablet,strip of 5 tablet dt,tablet,dt
135,136,AF 150 Tablet DT,strip of 1 Tablet DT,tablet,dt
150,151,Acogut 300 ER Tablet,strip of 10 tablet er,tablet,er
152,153,AB-Flo SR Tablet,strip of 10 tablet sr,tablet,sr
167,168,Alerfix Total Tablet SR,strip of 15 tablet sr,tablet,sr
186,187,Acivir 800 DT Tablet,strip of 5 tablet dt,tablet,dt
193,194,Ace-Proxyvon CR Tablet,strip of 10 tablet cr,tablet,cr
205,206,Azithral 250mg DT Tablet,strip of 10 tablet dt,tablet,dt



Total medicines with a release modifier: 21319


In [126]:
import re
import pandas as pd

FORMULATION_MODIFIERS = [
    "sr", "er", "xr", "cr", "mr",
    "dr", "dt", "md", "ir"
]

MODIFIER_PATTERN = re.compile(
    r"\b(" + "|".join(FORMULATION_MODIFIERS) + r")\b",
    flags=re.IGNORECASE
)


def extract_modifiers(value):
    if pd.isna(value):
        return tuple()

    matches = MODIFIER_PATTERN.findall(str(value))

    # Remove repeated tokens and standardize their case
    return tuple(sorted(set(
        match.lower() for match in matches
    )))


medicine_identity_df["name_modifiers"] = (
    medicine_identity_df["name"].map(extract_modifiers)
)

medicine_identity_df["pack_modifiers"] = (
    medicine_identity_df["pack_size"].map(extract_modifiers)
)

In [127]:
def classify_modifier_agreement(row):

    name_mods = set(row["name_modifiers"])
    pack_mods = set(row["pack_modifiers"])

    if name_mods and pack_mods:

        if name_mods == pack_mods:
            return "agreement"

        return "conflict"

    if name_mods:
        return "name_only"

    if pack_mods:
        return "pack_only"

    return "no_modifier"


medicine_identity_df["modifier_agreement"] = (
    medicine_identity_df.apply(
        classify_modifier_agreement,
        axis=1
    )
)

In [128]:
print("Modifier agreement summary:")

display(
    medicine_identity_df["modifier_agreement"]
    .value_counts()
)

print("\nConflicting records:")

modifier_conflicts = medicine_identity_df.loc[
    medicine_identity_df["modifier_agreement"] == "conflict",
    [
        "medicine_id",
        "name",
        "pack_size",
        "name_modifiers",
        "pack_modifiers",
    ]
]

print("Total conflicts:", len(modifier_conflicts))

display(modifier_conflicts.head(30))

Modifier agreement summary:


modifier_agreement
no_modifier    230076
agreement       18549
pack_only        2563
name_only        2515
conflict          270
Name: count, dtype: int64


Conflicting records:
Total conflicts: 270


,medicine_id,name,pack_size,name_modifiers,pack_modifiers
127,128,Acton-OR Tablet SR,strip of 10 tablet ir,['sr'],['ir']
869,870,Aim7 60 XR Tablet,strip of 15 tablet sr,['xr'],['sr']
1364,1365,Arflur CR Tablet,strip of 10 tablet sr,['cr'],['sr']
3243,3244,Angivex 2.6 SR Tablet,bottle of 30 tablet cr,['sr'],['cr']
5572,5573,Avgesic-CR Tablet,strip of 10 tablet sr,['cr'],['sr']
6975,6976,Acute CR 200mg/20mg Capsule SR,strip of 10 capsule sr,['cr' 'sr'],['sr']
13897,13898,Allervict-MD Tablet SR,strip of 10 tablet sr,['md' 'sr'],['sr']
18454,18455,Algibra D Capsule IR,strip of 10 capsule sr,['ir'],['sr']
25835,25836,Bupron SR 150 Tablet,strip of 10 tablet er,['sr'],['er']
27349,27350,Bacrite 20 XR Tablet,strip of 10 tablet er,['xr'],['er']


In [129]:
display(
    medicine_identity_df.loc[
        medicine_identity_df["name"].str.contains(
            "Acton-OR",
            case=False,
            na=False
        ),
        [
            "medicine_id",
            "name",
            "pack_size",
            "name_modifiers",
            "pack_modifiers",
            "modifier_agreement",
        ]
    ]
)

,medicine_id,name,pack_size,name_modifiers,pack_modifiers,modifier_agreement
127,128,Acton-OR Tablet SR,strip of 10 tablet ir,['sr'],['ir'],conflict
965,966,Acton-OR Plus+ Tablet,strip of 10 tablets,[],[],no_modifier


In [132]:
# Create temporary grouping columns
modifier_conflicts["name_modifiers_key"] = (
    modifier_conflicts["name_modifiers"]
    .apply(lambda x: " | ".join(map(str, x)) if isinstance(x, list)
           else str(x))
)

modifier_conflicts["pack_modifiers_key"] = (
    modifier_conflicts["pack_modifiers"]
    .apply(lambda x: " | ".join(map(str, x)) if isinstance(x, list)
           else str(x))
)

# Group conflict patterns
conflict_patterns = (
    modifier_conflicts
    .groupby(
        ["name_modifiers_key", "pack_modifiers_key"],
        dropna=False
    )
    .size()
    .reset_index(name="record_count")
    .sort_values("record_count", ascending=False)
)

display(conflict_patterns.head(30))

print("Distinct conflict patterns:", len(conflict_patterns))
print(
    "Total conflicting records:",
    conflict_patterns["record_count"].sum()
)

,name_modifiers_key,pack_modifiers_key,record_count
35,['xr'],['er'],84
32,['sr'],['er'],28
2,['cr'],['er'],26
26,['md'],['dt'],23
37,['xr'],['sr'],17
33,['sr'],['ir'],12
5,['cr'],['sr'],8
27,['mr' 'sr'],['sr'],6
13,['er' 'xr'],['er'],5
18,['er'],['sr'],5


Distinct conflict patterns: 38
Total conflicting records: 270


In [131]:
print("Modifiers found only in medicine names:")

name_only_records = medicine_identity_df.loc[
    medicine_identity_df["modifier_agreement"] == "name_only",
    ["medicine_id", "name", "pack_size", "name_modifiers"]
]

display(name_only_records.head(25))


print("Modifiers found only in pack descriptions:")

pack_only_records = medicine_identity_df.loc[
    medicine_identity_df["modifier_agreement"] == "pack_only",
    ["medicine_id", "name", "pack_size", "pack_modifiers"]
]

display(pack_only_records.head(25))

Modifiers found only in medicine names:


,medicine_id,name,pack_size,name_modifiers
85,86,Acemiz -MR Tablet,strip of 10 tablets,['mr']
116,117,Aceclo-MR Tablet,strip of 10 tablets,['mr']
142,143,Acenac-MR Tablet,strip of 10 tablets,['mr']
286,287,Algesia CR 200mg/20mg Capsule,strip of 10 capsules,['cr']
425,426,Acimol MR 100mg/325mg/250mg Tablet,strip of 10 tablets,['mr']
921,922,Anxit 0.5 SR Tablet,strip of 10 Tablet pr,['sr']
1005,1006,Aldigesic MR Tablet,strip of 10 tablets,['mr']
1198,1199,Acenext MR 100mg/325mg/250mg Tablet,strip of 10 tablets,['mr']
1199,1200,Aceloflam MR Tablet,strip of 10 tablets,['mr']
1229,1230,Alcinac MR Tablet,strip of 10 tablets,['mr']


Modifiers found only in pack descriptions:


,medicine_id,name,pack_size,pack_modifiers
1413,1414,Alerab-DSR Capsule,strip of 10 capsule sr,['sr']
1494,1495,Arorab-DSR Capsule,strip of 10 capsule sr,['sr']
1518,1519,Aciban DSR Capsule,strip of 10 capsule sr,['sr']
1988,1989,Abmetop 50 XL Tablet,strip of 10 tablet er,['er']
2207,2208,Amazeo OD 400 Tablet,strip of 10 tablet sr,['sr']
2292,2293,Aspan-DSR Capsule,strip of 10 capsule sr,['sr']
2497,2498,Anurab DSR Capsule,strip of 10 capsule sr,['sr']
2532,2533,Asterab DSR Capsule,strip of 10 capsule sr,['sr']
2544,2545,Amrizol-DSR Capsule,strip of 10 capsule sr,['sr']
2619,2620,Aclomet-XL 25 Tablet,strip of 10 tablet er,['er']


In [133]:
from collections import Counter

name_modifier_counts = Counter(
    modifier
    for modifiers in medicine_identity_df["name_modifiers"]
    for modifier in modifiers
)

pack_modifier_counts = Counter(
    modifier
    for modifiers in medicine_identity_df["pack_modifiers"]
    for modifier in modifiers
)

modifier_source_summary = pd.DataFrame({
    "medicine_name": pd.Series(name_modifier_counts),
    "pack_description": pd.Series(pack_modifier_counts)
}).fillna(0).astype(int)

display(modifier_source_summary)

,medicine_name,pack_description
cr,829,728
dr,182,126
dt,4620,4649
er,1951,2235
ir,62,63
md,1887,1836
mr,2357,312
sr,9121,11243
xr,353,190


In [134]:
designation_pattern = (
    r"\b(DSR|XL|PR|OD|EC|LA|CR|SR|ER|XR|MR|DR|"
    r"DT|MD|IR|TR|DRR)\b"
)

designation_records = medicine_identity_df.loc[
    medicine_identity_df["name"].str.contains(
        designation_pattern,
        case=False,
        na=False,
        regex=True
    ),
    [
        "medicine_id",
        "name",
        "pack_size",
        "name_modifiers",
        "pack_modifiers",
    ]
].copy()

print("Records containing these designations:", len(designation_records))

display(designation_records.head(60))

C:\Users\HP\AppData\Local\Temp\ipykernel_8644\3963058862.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  medicine_identity_df["name"].str.contains(


Records containing these designations: 26378


,medicine_id,name,pack_size,name_modifiers,pack_modifiers
22,23,Altraday Capsule SR,strip of 10 capsule sr,['sr'],['sr']
37,38,Alfoo 10mg Tablet PR,strip of 30 Tablet pr,[],[]
73,74,AB Phylline SR 200 Tablet,strip of 10 tablet sr,['sr'],['sr']
83,84,Angispan - TR 2.5mg Capsule,bottle of 25 capsule tr,[],[]
85,86,Acemiz -MR Tablet,strip of 10 tablets,['mr'],[]
89,90,Acivir 400 DT Tablet,strip of 5 tablet dt,['dt'],['dt']
116,117,Aceclo-MR Tablet,strip of 10 tablets,['mr'],[]
127,128,Acton-OR Tablet SR,strip of 10 tablet ir,['sr'],['ir']
135,136,AF 150 Tablet DT,strip of 1 Tablet DT,['dt'],['dt']
137,138,Amaryl M 1mg Tablet PR,strip of 20 Tablet pr,[],[]


In [135]:
pack_designation_records = medicine_identity_df.loc[
    medicine_identity_df["pack_size"].str.contains(
        r"\b(DSR|XL|PR|OD|EC|LA|TR)\b",
        case=False,
        na=False,
        regex=True
    ),
    [
        "medicine_id",
        "name",
        "pack_size",
    ]
]

print("Pack descriptions with additional designations:",
      len(pack_designation_records))

display(pack_designation_records.head(40))

C:\Users\HP\AppData\Local\Temp\ipykernel_8644\4163372794.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  medicine_identity_df["pack_size"].str.contains(


Pack descriptions with additional designations: 1799


,medicine_id,name,pack_size
37,38,Alfoo 10mg Tablet PR,strip of 30 Tablet pr
83,84,Angispan - TR 2.5mg Capsule,bottle of 25 capsule tr
137,138,Amaryl M 1mg Tablet PR,strip of 20 Tablet pr
146,147,Amaryl M 2mg Tablet PR,strip of 20 Tablet pr
187,188,Alfusin Tablet PR,strip of 15 Tablet pr
203,204,Angiplat 2.5 Capsule TR,bottle of 25 capsule tr
211,212,Azulix 2 MF Tablet PR,strip of 15 Tablet pr
213,214,Azulix 1 MF Tablet PR,strip of 15 Tablet pr
283,284,Amaryl M Forte 2mg Tablet PR,strip of 15 Tablet pr
389,390,Amtas-M 50 Tablet PR,strip of 15 Tablet pr


In [136]:
ADDITIONAL_DESIGNATIONS = [
    "dsr", "xl", "pr", "od", "ec", "la", "tr"
]

ADDITIONAL_DESIGNATION_PATTERN = re.compile(
    r"\b(" + "|".join(ADDITIONAL_DESIGNATIONS) + r")\b",
    flags=re.IGNORECASE
)


def extract_additional_designations(value):
    if pd.isna(value):
        return tuple()

    matches = ADDITIONAL_DESIGNATION_PATTERN.findall(
        str(value)
    )

    return tuple(sorted(set(
        match.lower() for match in matches
    )))


medicine_identity_df["name_designations"] = (
    medicine_identity_df["name"].map(
        extract_additional_designations
    )
)

medicine_identity_df["pack_designations"] = (
    medicine_identity_df["pack_size"].map(
        extract_additional_designations
    )
)

In [137]:
from collections import Counter

name_designation_counts = Counter(
    item
    for items in medicine_identity_df["name_designations"]
    for item in items
)

pack_designation_counts = Counter(
    item
    for items in medicine_identity_df["pack_designations"]
    for item in items
)

designation_summary = pd.DataFrame({
    "medicine_name": pd.Series(name_designation_counts),
    "pack_description": pd.Series(pack_designation_counts)
}).fillna(0).astype(int)

display(designation_summary)

,medicine_name,pack_description
dsr,2368,0
ec,61,0
la,127,0
od,785,0
pr,1075,1448
tr,124,45
xl,882,306


In [139]:
# Helper function to safely compare designation values
def normalize_designations(value):
    if value is None:
        return set()

    if isinstance(value, (list, tuple, set)):
        return set(value)

    if hasattr(value, "tolist"):
        value = value.tolist()
        return set(value) if isinstance(value, list) else {value}

    return set()


# Identify designation disagreements
designation_conflicts = medicine_identity_df.loc[
    medicine_identity_df.apply(
        lambda row: (
            normalize_designations(row["name_designations"])
            != normalize_designations(row["pack_designations"])
        ) and (
            len(normalize_designations(row["name_designations"])) > 0
            or len(normalize_designations(row["pack_designations"])) > 0
        ),
        axis=1
    ),
    [
        "medicine_id",
        "name",
        "pack_size",
        "name_designations",
        "pack_designations"
    ]
]

print(
    "Additional designation disagreements:",
    len(designation_conflicts)
)

display(designation_conflicts.head(30))

Additional designation disagreements: 4268


,medicine_id,name,pack_size,name_designations,pack_designations
138,139,Azithral XL 200 Liquid,bottle of 30 ml Oral Suspension,['xl'],[]
399,400,Azee XL 200mg Dry Syrup Peppermint,bottle of 30 ml Dry Syrup,['xl'],[]
632,633,Acotrust OD 300mg Tablet ER,strip of 10 tablet er,['od'],[]
723,724,Azithral-XL 100 Liquid,bottle of 30 ml Oral Suspension,['xl'],[]
808,809,Anleo-DSR Capsule,strip of 10 capsule pr,['dsr'],['pr']
868,869,Aziwok-XL 200 Liquid,bottle of 30 ml Suspension,['xl'],[]
880,881,Addwize OD 18mg Tablet,strip of 10 tablets,['od'],[]
921,922,Anxit 0.5 SR Tablet,strip of 10 Tablet pr,[],['pr']
978,979,Azee XL 100mg Dry Syrup Peppermint,bottle of 30 ml Dry Syrup,['xl'],[]
990,991,Amazeo OD 100 Tablet SR,strip of 10 tablet sr,['od'],[]


In [140]:
import pandas as pd

def select_verified_modifier(row):

    if row["modifier_agreement"] == "agreement":
        return row["name_modifiers"]

    return tuple()


medicine_identity_df["formulation_modifier_final"] = (
    medicine_identity_df.apply(
        select_verified_modifier,
        axis=1
    )
)

medicine_identity_df["formulation_review_required"] = (
    medicine_identity_df["modifier_agreement"].isin(
        ["conflict", "name_only", "pack_only"]
    )
)

medicine_identity_df["formulation_status"] = (
    medicine_identity_df["modifier_agreement"].map({
        "agreement": "verified_from_both_sources",
        "no_modifier": "unspecified",
        "name_only": "name_only_unverified",
        "pack_only": "pack_only_unverified",
        "conflict": "source_conflict"
    })
)

print("Final formulation status:")
display(
    medicine_identity_df["formulation_status"].value_counts()
)

print(
    "\nRecords requiring formulation review:",
    medicine_identity_df["formulation_review_required"].sum()
)

Final formulation status:


formulation_status
unspecified                   230076
verified_from_both_sources     18549
pack_only_unverified            2563
name_only_unverified            2515
source_conflict                  270
Name: count, dtype: int64


Records requiring formulation review: 5348


In [141]:
medicine_identity_df["additional_designations"] = (
    medicine_identity_df.apply(
        lambda row: tuple(sorted(
            set(row["name_designations"]) |
            set(row["pack_designations"])
        )),
        axis=1
    )
)

In [142]:
display(
    medicine_identity_df[
        [
            "medicine_id",
            "name",
            "pack_size",
            "formulation_modifier_final",
            "formulation_status",
            "formulation_review_required",
            "additional_designations"
        ]
    ].head(30)
)

,medicine_id,name,pack_size,formulation_modifier_final,formulation_status,formulation_review_required,additional_designations
0,1,Augmentin 625 Duo Tablet,strip of 10 tablets,(),unspecified,False,()
1,2,Azithral 500 Tablet,strip of 5 tablets,(),unspecified,False,()
2,3,Ascoril LS Syrup,bottle of 100 ml Syrup,(),unspecified,False,()
3,4,Allegra 120mg Tablet,strip of 10 tablets,(),unspecified,False,()
4,5,Avil 25 Tablet,strip of 15 tablets,(),unspecified,False,()
5,6,Allegra-M Tablet,strip of 10 tablets,(),unspecified,False,()
6,7,Amoxyclav 625 Tablet,strip of 10 tablets,(),unspecified,False,()
7,8,Azee 500 Tablet,strip of 5 tablets,(),unspecified,False,()
8,9,Atarax 25mg Tablet,strip of 15 tablets,(),unspecified,False,()
9,10,Ascoril D Plus Syrup Sugar Free,bottle of 100 ml Syrup,(),unspecified,False,()


In [143]:
print("Total medicines:", len(medicine_identity_df))

print(
    "Missing medicine IDs:",
    medicine_identity_df["medicine_id"].isna().sum()
)

print(
    "Duplicate medicine IDs:",
    medicine_identity_df["medicine_id"].duplicated().sum()
)

Total medicines: 253973
Missing medicine IDs: 0
Duplicate medicine IDs: 0


In [144]:
# Check which columns are available in the identity dataframe

print("Identity dataframe columns:")
print(medicine_identity_df.columns.tolist())

print("\nRelevant column checks:")

required_columns = [
    "medicine_id",
    "name",
    "pack_size",
    "canonical_ingredient_signature",
    "ingredient_count",
    "has_duplicate_component",
    "has_unparsed_strength",
    "formulation_modifier_final",
    "formulation_status",
    "formulation_review_required",
]

for column in required_columns:
    print(f"{column}: {column in medicine_identity_df.columns}")

Identity dataframe columns:
['medicine_id', 'name', 'price', 'is_discontinued', 'manufacturer', 'dosage_form', 'pack_size', 'composition1', 'composition2', 'price_original', 'price_status', 'parsed_ingredients', 'ingredient_count', 'dosage_form_normalized', 'formulation_modifier', 'formulation_detail', 'formulation_status', 'canonical_ingredient_signature', 'parsed_ingredient_count', 'has_duplicate_component', 'has_unparsed_strength', 'has_unparsed_normalization', 'name_modifiers', 'pack_modifiers', 'modifier_agreement', 'name_designations', 'pack_designations', 'formulation_modifier_final', 'formulation_review_required', 'additional_designations']

Relevant column checks:
medicine_id: True
name: True
pack_size: True
canonical_ingredient_signature: True
ingredient_count: True
has_duplicate_component: True
has_unparsed_strength: True
formulation_modifier_final: True
formulation_status: True
formulation_review_required: True


In [145]:
form_columns = [
    column for column in medicine_identity_df.columns
    if "form" in column.lower()
]

print("Possible product-form columns:", form_columns)

Possible product-form columns: ['dosage_form', 'dosage_form_normalized', 'formulation_modifier', 'formulation_detail', 'formulation_status', 'formulation_modifier_final', 'formulation_review_required']


In [147]:
print([
    column for column in clean_df.columns
    if "form" in column.lower()
])

['dosage_form', 'dosage_form_normalized', 'formulation_modifier', 'formulation_detail', 'formulation_status']


In [148]:
import json
import hashlib
import pandas as pd


def normalize_modifier_tuple(value):
    """
    Convert a modifier tuple/list into a sorted,
    consistent tuple of lowercase strings.
    """
    if value is None:
        return tuple()

    if isinstance(value, (tuple, list)):
        return tuple(sorted(
            str(item).strip().lower()
            for item in value
            if str(item).strip()
        ))

    return tuple()


def create_identity_payload(row):
    """
    Create the information used to identify a medicine.
    Pack quantity, brand and manufacturer are excluded.
    """

    modifier = normalize_modifier_tuple(
        row["formulation_modifier_final"]
    )

    payload = {
        "ingredients": row["canonical_ingredient_signature"],
        "product_form": str(row["dosage_form_normalized"]).strip().lower(),
        "formulation_status": row["formulation_status"],
        "formulation_modifier": modifier,
    }

    return json.dumps(
        payload,
        sort_keys=True,
        separators=(",", ":")
    )


medicine_identity_df["canonical_identity_payload"] = (
    medicine_identity_df.apply(
        create_identity_payload,
        axis=1
    )
)


def create_identity_hash(payload):
    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()


medicine_identity_df["canonical_identity_key"] = (
    medicine_identity_df["canonical_identity_payload"].map(
        create_identity_hash
    )
)

In [149]:
medicine_identity_df["match_eligible"] = (
    medicine_identity_df["has_unparsed_strength"].eq(False)
    &
    medicine_identity_df["has_unparsed_normalization"].eq(False)
    &
    medicine_identity_df["has_duplicate_component"].eq(False)
    &
    medicine_identity_df["dosage_form_normalized"].notna()
    &
    medicine_identity_df["dosage_form_normalized"].ne("unknown")
    &
    medicine_identity_df["formulation_review_required"].eq(False)
    &
    medicine_identity_df["formulation_status"].isin([
        "unspecified",
        "verified_from_both_sources"
    ])
)

In [150]:
def get_exclusion_reasons(row):

    reasons = []

    if row["has_unparsed_strength"]:
        reasons.append("unparsed_strength")

    if row["has_unparsed_normalization"]:
        reasons.append("unparsed_normalization")

    if row["has_duplicate_component"]:
        reasons.append("duplicate_component_review")

    if pd.isna(row["dosage_form_normalized"]):
        reasons.append("missing_product_form")

    elif row["dosage_form_normalized"] == "unknown":
        reasons.append("unknown_product_form")

    if row["formulation_review_required"]:
        reasons.append("formulation_review")

    if row["formulation_status"] not in [
        "unspecified",
        "verified_from_both_sources"
    ]:
        reasons.append("unverified_formulation")

    return reasons


medicine_identity_df["match_exclusion_reason"] = (
    medicine_identity_df.apply(
        get_exclusion_reasons,
        axis=1
    )
)

In [151]:
print("Total medicine records:",
      len(medicine_identity_df))

print("\nMatching eligibility:")
display(
    medicine_identity_df["match_eligible"].value_counts()
)

print("\nEligibility percentage:")
print(
    round(
        medicine_identity_df["match_eligible"].mean() * 100,
        2
    ),
    "%"
)

print("\nExclusion reasons:")
exclusion_counts = (
    medicine_identity_df["match_exclusion_reason"]
    .explode()
    .dropna()
    .value_counts()
)

display(exclusion_counts)

print("\nUnique identity keys:",
      medicine_identity_df["canonical_identity_key"].nunique())

print("\nMissing identity keys:",
      medicine_identity_df["canonical_identity_key"].isna().sum())

print("\nMissing medicine IDs:",
      medicine_identity_df["medicine_id"].isna().sum())

Total medicine records: 253973

Matching eligibility:


match_eligible
True     244649
False      9324
Name: count, dtype: int64


Eligibility percentage:
96.33 %

Exclusion reasons:


match_exclusion_reason
formulation_review            5348
unverified_formulation        5348
unparsed_strength             3872
unparsed_normalization        3872
unknown_product_form           143
duplicate_component_review       3
Name: count, dtype: int64


Unique identity keys: 16647

Missing identity keys: 0

Missing medicine IDs: 0


In [152]:
inspection_names = [
    "Augmentin 625 Duo Tablet",
    "Amoxyclav 625 Tablet"
]

display(
    medicine_identity_df.loc[
        medicine_identity_df["name"].isin(inspection_names),
        [
            "medicine_id",
            "name",
            "canonical_ingredient_signature",
            "dosage_form_normalized",
            "formulation_status",
            "canonical_identity_key",
            "match_eligible"
        ]
    ]
)

,medicine_id,name,canonical_ingredient_signature,dosage_form_normalized,formulation_status,canonical_identity_key,match_eligible
0,1,Augmentin 625 Duo Tablet,"[{""basis"":null,""denominator_unit"":null,""denomi...",tablet,unspecified,f0baddc4174e9f7b004b451273047e717df9c252d51a1a...,True
6,7,Amoxyclav 625 Tablet,"[{""basis"":null,""denominator_unit"":null,""denomi...",tablet,unspecified,f0baddc4174e9f7b004b451273047e717df9c252d51a1a...,True


In [153]:
import re
import pandas as pd

FORMULATION_MODIFIERS = [
    "sr", "er", "xr", "cr", "mr",
    "dr", "dt", "md", "ir"
]

MODIFIER_PATTERN = re.compile(
    r"\b(" + "|".join(FORMULATION_MODIFIERS) + r")\b",
    flags=re.IGNORECASE
)


def extract_modifiers(value):
    if pd.isna(value):
        return tuple()

    matches = MODIFIER_PATTERN.findall(str(value))

    # Remove repeated tokens and standardize their case
    return tuple(sorted(set(
        match.lower() for match in matches
    )))


medicine_identity_df["name_modifiers"] = (
    medicine_identity_df["name"].map(extract_modifiers)
)

medicine_identity_df["pack_modifiers"] = (
    medicine_identity_df["pack_size"].map(extract_modifiers)
)

In [156]:
identity_groups = (
    medicine_identity_df
    .groupby(
        "canonical_ingredient_signature",
        dropna=False
    )
    .agg(
        different_forms=("dosage_form_normalized", "nunique"),

        different_formulation_statuses=(
            "formulation_status",
            "nunique"
        ),

        medicine_count=("medicine_id", "nunique")
    )
    .reset_index()
)

print("Identity groups created:", len(identity_groups))

display(identity_groups.head(10))

Identity groups created: 11242


,canonical_ingredient_signature,different_forms,different_formulation_statuses,medicine_count
0,"[{""basis"":""v/v"",""denominator_unit"":null,""denom...",1,1,1
1,"[{""basis"":""v/v"",""denominator_unit"":null,""denom...",1,1,1
2,"[{""basis"":""v/v"",""denominator_unit"":null,""denom...",1,1,1
3,"[{""basis"":""v/v"",""denominator_unit"":null,""denom...",1,1,1
4,"[{""basis"":""v/v"",""denominator_unit"":null,""denom...",1,1,1
5,"[{""basis"":""v/v"",""denominator_unit"":null,""denom...",1,1,1
6,"[{""basis"":""v/v"",""denominator_unit"":null,""denom...",1,1,5
7,"[{""basis"":""v/v"",""denominator_unit"":null,""denom...",1,1,2
8,"[{""basis"":""v/v"",""denominator_unit"":null,""denom...",1,1,1
9,"[{""basis"":""v/v"",""denominator_unit"":null,""denom...",1,1,1


In [157]:
# Groups that contain more than one product form or
# more than one formulation status should not occur.

inconsistent_groups = identity_groups.loc[
    (identity_groups["different_forms"] > 1)
    |
    (identity_groups["different_formulation_statuses"] > 1)
]

print("Inconsistent identity groups:", len(inconsistent_groups))
display(inconsistent_groups.head(20))

Inconsistent identity groups: 3041


,canonical_ingredient_signature,different_forms,different_formulation_statuses,medicine_count
25,"[{""basis"":""w/v"",""denominator_unit"":null,""denom...",2,1,2
32,"[{""basis"":""w/v"",""denominator_unit"":null,""denom...",2,1,3
35,"[{""basis"":""w/v"",""denominator_unit"":null,""denom...",2,1,17
40,"[{""basis"":""w/v"",""denominator_unit"":null,""denom...",3,1,3
42,"[{""basis"":""w/v"",""denominator_unit"":null,""denom...",2,1,15
59,"[{""basis"":""w/v"",""denominator_unit"":null,""denom...",4,1,49
61,"[{""basis"":""w/v"",""denominator_unit"":null,""denom...",2,1,9
64,"[{""basis"":""w/v"",""denominator_unit"":null,""denom...",2,1,23
67,"[{""basis"":""w/v"",""denominator_unit"":null,""denom...",3,1,12
69,"[{""basis"":""w/v"",""denominator_unit"":null,""denom...",2,1,3


In [160]:
eligible_df = medicine_identity_df.loc[
    medicine_identity_df["match_eligible"] == True
].copy()

print("Total medicines:", len(medicine_identity_df))
print("Eligible medicines:", len(eligible_df))
print("Excluded medicines:", len(medicine_identity_df) - len(eligible_df))

display(eligible_df.head())

Total medicines: 253973
Eligible medicines: 244649
Excluded medicines: 9324


,medicine_id,name,price,is_discontinued,manufacturer,dosage_form,pack_size,composition1,composition2,price_original,...,modifier_agreement,name_designations,pack_designations,formulation_modifier_final,formulation_review_required,additional_designations,canonical_identity_payload,canonical_identity_key,match_eligible,match_exclusion_reason
0,1,Augmentin 625 Duo Tablet,223.42,False,Glaxo SmithKline Pharmaceuticals Ltd,allopathy,strip of 10 tablets,Amoxycillin (500mg),Clavulanic Acid (125mg),223.42,...,no_modifier,[],[],(),False,(),"{""formulation_modifier"":[],""formulation_status...",f0baddc4174e9f7b004b451273047e717df9c252d51a1a...,True,[]
1,2,Azithral 500 Tablet,132.36,False,Alembic Pharmaceuticals Ltd,allopathy,strip of 5 tablets,Azithromycin (500mg),<NA>,132.36,...,no_modifier,[],[],(),False,(),"{""formulation_modifier"":[],""formulation_status...",7e67eb066fc55ada3261ad1f0f3082cb56cb5e4973291e...,True,[]
2,3,Ascoril LS Syrup,118.00,False,Glenmark Pharmaceuticals Ltd,allopathy,bottle of 100 ml Syrup,Ambroxol (30mg/5ml),Levosalbutamol (1mg/5ml),118.00,...,no_modifier,[],[],(),False,(),"{""formulation_modifier"":[],""formulation_status...",333dd7ce27d4557c9f0e4f9710664cede8a04605b68bfd...,True,[]
3,4,Allegra 120mg Tablet,218.81,False,Sanofi India Ltd,allopathy,strip of 10 tablets,Fexofenadine (120mg),<NA>,218.81,...,no_modifier,[],[],(),False,(),"{""formulation_modifier"":[],""formulation_status...",83068b16fefc32c2fbef4576991095666d24975f889db5...,True,[]
4,5,Avil 25 Tablet,10.96,False,Sanofi India Ltd,allopathy,strip of 15 tablets,Pheniramine (25mg),<NA>,10.96,...,no_modifier,[],[],(),False,(),"{""formulation_modifier"":[],""formulation_status...",fb7492a0a5a3e86e7a34a60d5a41b47c6a2ca8acb1abf2...,True,[]


In [164]:
# Find identity keys containing multiple medicine records
multi_keys = (
    identity_groups.loc[
        identity_groups["medicine_count"] > 1,
        "canonical_identity_key"
    ]
    .head(10)
)

display(
    eligible_df.loc[
        eligible_df["canonical_identity_key"].isin(multi_keys),
        [
            "medicine_id",
            "name",
            "manufacturer",
            "price",
            "pack_size",
            "dosage_form_normalized",
            "formulation_modifier_final",
            "canonical_identity_key"
        ]
    ]
    .sort_values("canonical_identity_key")
)

,medicine_id,name,manufacturer,price,pack_size,dosage_form_normalized,formulation_modifier_final,canonical_identity_key
68706,68707,Doxopar 50mg Injection,Parenteral Drugs India Ltd,1200.00,vial of 1 Injection,injection,(),0000a0ccc9ca17d5b67ef62d3ade72e5e071bb18e99e96...
69955,69956,Doxoson 50mg Injection,Resonance Laboratories Pvt Ltd,1050.00,vial of 1 Injection,injection,(),0000a0ccc9ca17d5b67ef62d3ade72e5e071bb18e99e96...
110184,110185,I-Dox 50mg Injection,Getwell Pharmaceutical Pvt Ltd,19800.00,vial of 25 ml Injection,injection,(),0000a0ccc9ca17d5b67ef62d3ade72e5e071bb18e99e96...
124311,124312,Lippod 50mg Injection,Celon Laboratories Ltd,18200.00,vial of 25 ml Injection,injection,(),0000a0ccc9ca17d5b67ef62d3ade72e5e071bb18e99e96...
128428,128429,Lipisol 50mg Injection,Alkem Laboratories Ltd,8163.26,vial of 10 ml Injection,injection,(),0000a0ccc9ca17d5b67ef62d3ade72e5e071bb18e99e96...
...,...,...,...,...,...,...,...,...
146439,146440,MS Beta 6mg Tablet,Eisai Pharmaceuticals India Pvt Ltd,64.00,strip of 10 tablets,tablet,(),0045eeb0bf9072fd83054b27e33f7aed04cffeb7e9f916...
173786,173787,Proxyvon Plus Capsule,Dr Reddy's Laboratories Ltd,51.00,strip of 8 capsules,capsule,(),004ad9fdd5998649dbb798371e974e192af353a985ba80...
220227,220228,Tram Proxyvon 325 mg/37.5 mg Capsule,Wockhardt Ltd,24.00,strip of 8 capsules,capsule,(),004ad9fdd5998649dbb798371e974e192af353a985ba80...
229478,229479,Tronus-P 37.5mg/325mg Capsule,Cronus Biotech Ltd,55.00,strip of 10 capsules,capsule,(),004ad9fdd5998649dbb798371e974e192af353a985ba80...


In [165]:
eligible_df = medicine_identity_df.loc[
    medicine_identity_df["match_eligible"]
].copy()

identity_groups = (
    eligible_df
    .groupby("canonical_identity_key")
    .agg(
        medicine_count=("medicine_id", "count"),
        different_ingredient_signatures=(
            "canonical_ingredient_signature", "nunique"
        ),
        different_forms=("dosage_form_normalized", "nunique"),
        different_formulation_statuses=(
            "formulation_status", "nunique"
        ),
        different_modifiers=(
            "formulation_modifier_final",
            lambda x: x.astype(str).nunique()
        ),
        different_names=("name", "nunique"),
        different_manufacturers=("manufacturer", "nunique")
    )
    .reset_index()
)

print("Eligible medicines:", len(eligible_df))

print("Unique eligible identity groups:", len(identity_groups))

print(
    "\nGroups with multiple medicine records:",
    (identity_groups["medicine_count"] > 1).sum()
)

print(
    "\nGroups with different ingredient signatures:",
    (identity_groups["different_ingredient_signatures"] > 1).sum()
)

print(
    "Groups with different product forms:",
    (identity_groups["different_forms"] > 1).sum()
)

print(
    "Groups with different formulation statuses:",
    (identity_groups["different_formulation_statuses"] > 1).sum()
)

print(
    "Groups with different modifiers:",
    (identity_groups["different_modifiers"] > 1).sum()
)

Eligible medicines: 244649
Unique eligible identity groups: 14503

Groups with multiple medicine records: 7960

Groups with different ingredient signatures: 0
Groups with different product forms: 0
Groups with different formulation statuses: 0
Groups with different modifiers: 306


In [166]:
largest_groups = identity_groups.sort_values(
    "medicine_count",
    ascending=False
).head(10)

display(largest_groups)

,canonical_identity_key,medicine_count,different_ingredient_signatures,different_forms,different_formulation_statuses,different_modifiers,different_names,different_manufacturers
12185,d76eb151a8693eca06f223940804115bd96e6d5b8ba265...,3853,1,1,1,1,3830,2249
13620,f0baddc4174e9f7b004b451273047e717df9c252d51a1a...,2581,1,1,1,1,2531,2052
3220,38173dad983475c8a097272297a23c470508e3d62e7089...,2294,1,1,1,1,2270,2099
3095,363c3c15bece9d23b77a3c900fb124788d5ee0aff45918...,1946,1,1,1,1,1933,1746
7229,7e67eb066fc55ada3261ad1f0f3082cb56cb5e4973291e...,1946,1,1,1,1,1841,1721
12208,d7efcf69eaafe1eef692876f0edadf9b261819911c8161...,1902,1,1,1,1,1846,1707
7549,8467af701305fa279dd29d786d13abe5a001065c7b20a5...,1868,1,1,1,1,1827,1686
5278,5c60d0408129bc9aa52cd4013317df28711b2e194f9b74...,1644,1,1,1,1,1599,1477
9545,a8a29dad660956c42948801c2e790d5a61329a98c884e4...,1532,1,1,1,1,1495,1407
12136,d6a0b1a31746a13f8183a0c2e0f41feefbabef113b3f10...,1524,1,1,1,1,1462,1359


In [167]:
top_keys = largest_groups["canonical_identity_key"]

display(
    eligible_df.loc[
        eligible_df["canonical_identity_key"].isin(top_keys),
        [
            "medicine_id",
            "name",
            "manufacturer",
            "price",
            "pack_size",
            "dosage_form_normalized",
            "formulation_status",
            "formulation_modifier_final",
            "canonical_identity_key"
        ]
    ].sort_values(
        ["canonical_identity_key", "price"]
    )
)

,medicine_id,name,manufacturer,price,pack_size,dosage_form_normalized,formulation_status,formulation_modifier_final,canonical_identity_key
13640,13641,Aaltranido O 200mg/500mg Tablet,Aaltramed Healthcare Ltd,6.90,strip of 1 Tablet,tablet,unspecified,(),363c3c15bece9d23b77a3c900fb124788d5ee0aff45918...
164759,164760,Otex OZ Tablet,Biocare Research India Pvt Ltd,8.25,strip of 1 Tablet,tablet,unspecified,(),363c3c15bece9d23b77a3c900fb124788d5ee0aff45918...
167953,167954,Oflavid OZ Tablet,Leeford Healthcare Ltd,17.50,strip of 6 tablets,tablet,unspecified,(),363c3c15bece9d23b77a3c900fb124788d5ee0aff45918...
148630,148631,Maidrof 200mg/500mg Tablet,Maiden Pharmaceutical Ltd,19.00,strip of 10 tablets,tablet,unspecified,(),363c3c15bece9d23b77a3c900fb124788d5ee0aff45918...
8753,8754,Alfox OZ 200 mg/500 mg Tablet,Alliaance Biotech,20.00,strip of 10 tablets,tablet,unspecified,(),363c3c15bece9d23b77a3c900fb124788d5ee0aff45918...
...,...,...,...,...,...,...,...,...,...
339,340,Augpen LB 625 Tablet,Zuventus Healthcare Ltd,439.80,strip of 10 tablets,tablet,unspecified,(),f0baddc4174e9f7b004b451273047e717df9c252d51a1a...
217902,217903,Themibact CV 500mg/125mg Tablet,Themis Medicare Ltd,520.00,strip of 10 tablets,tablet,unspecified,(),f0baddc4174e9f7b004b451273047e717df9c252d51a1a...
40162,40163,Canmox 500mg/125mg Tablet,Forgo Pharmaceuticals Pvt Ltd,630.00,strip of 18 tablets,tablet,unspecified,(),f0baddc4174e9f7b004b451273047e717df9c252d51a1a...
66815,66816,DA 500mg/125mg Tablet,Abatis Holistic,975.00,strip of 30 tablets,tablet,unspecified,(),f0baddc4174e9f7b004b451273047e717df9c252d51a1a...


In [172]:
fresh_identity_groups["normalized_modifier_count"] = (
    eligible_df
    .assign(
        normalized_modifier=eligible_df[
            "formulation_modifier_final"
        ].map(normalize_modifier_tuple)
    )
    .groupby("canonical_identity_key")[
        "normalized_modifier"
    ]
    .nunique()
    .values
)

print(
    "Groups with different normalized modifiers:",
    (
        fresh_identity_groups[
            "normalized_modifier_count"
        ] > 1
    ).sum()
)

Groups with different normalized modifiers: 0


In [178]:
from pathlib import Path

processed_folder = Path("../data/processed")
processed_folder.mkdir(parents=True, exist_ok=True)


output_path = processed_folder / "medicines_cleaned.parquet"

clean_df.to_parquet(output_path, index=False)

print("Cleaned dataset saved successfully!")
print("Location:", output_path.resolve())
print("Shape:", clean_df.shape)

Cleaned dataset saved successfully!
Location: D:\Engsci_Study_materials\Subject_Index\Projects\cheaper_medicine_finder\data\processed\medicines_cleaned.parquet
Shape: (253973, 17)
